# Lanjut E17E · Konfirmasi lima fold sampai evaluasi test

Buka runtime T4, hubungkan Drive yang menyimpan study pencarian E17E, lalu
Run all. Notebook memulihkan database, menguji lima kandidat terbaik dan
baseline pada lima fold, memeriksa stabilitas sesuai protokol E17E, mengunci
pilihan dari validasi, lalu refit dan menghitung test.

Study harus sudah mencapai 60 trial COMPLETE atau PRUNED. Ranking Excel
dipakai sebagai ringkasan; pemulihan memerlukan folder study asli di Drive.
Log menampilkan study yang dipilih dan jumlah trial sebelum mulai konfirmasi.
Jika snapshot study belum ditemukan, tahap ini berhenti sebelum training.
Versi runtime, dataset, split dan konfigurasi perlu cocok dengan study asal.

Hasil final berupa ZIP 04–05, berisi empat metrik per trait untuk OOF/test,
prediksi, konfigurasi terpilih, riwayat training, dan tabel pencarian.
Tombol unduhan muncul setelah laporan diperiksa. Hasil tersimpan di Drive.
Jika runtime putus, Run all memakai ulang trial, fold dan hasil test yang selesai.
Fit yang terputus dimulai ulang dari awal.

Notebook ini merupakan peluncur tahap dari E17E. Identitas kode training dan
kontrak study berasal dari notebook pencarian asli. Metadata peluncur mencatat
checksum sumber dan seluruh cell kode peluncur.


## 1 · Setup pustaka pada kernel Colab

In [ ]:
# Jalankan pada runtime Colab baru, sebelum mengimpor pustaka numerik/model.
import sys
import subprocess
from importlib import metadata, util
from packaging.requirements import Requirement

# Pertahankan Torch/CUDA, NumPy, pandas, SciPy dan sklearn bawaan Colab.
# Mengganti pustaka numerik yang sudah dimuat kernel dapat menyebabkan konflik ABI.
CORE_REQUIREMENTS = {
    'torch': 'torch>=2.6', 'numpy': 'numpy>=1.26,<3',
    'pandas': 'pandas>=2,<3', 'scipy': 'scipy>=1.10,<2',
    'scikit-learn': 'scikit-learn>=1.2,<2',
}
for package, requirement in CORE_REQUIREMENTS.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError('Gunakan runtime Google Colab baru; pustaka dasar belum ada: '+package) from error
    if not Requirement(requirement).specifier.contains(installed):
        raise RuntimeError(requirement+' diperlukan; runtime ini memiliki '+installed+'. Pilih runtime Colab baru.')

# Tokenizers 0.19 tidak menyediakan wheel Python 3.13; tetap satu versi per runtime.
TRANSFORMERS_REQUIREMENT = 'transformers==4.48.3' if sys.version_info >= (3, 13) else 'transformers==4.44.2'
EXTRA_REQUIREMENTS = [TRANSFORMERS_REQUIREMENT, 'openpyxl>=3.1,<4', 'optuna==4.6.0']
to_install = []
for requirement in EXTRA_REQUIREMENTS:
    req = Requirement(requirement)
    try:
        installed = metadata.version(req.name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not req.specifier.contains(installed):
        if req.name in sys.modules:
            raise RuntimeError(req.name+' sudah diimpor dengan versi berbeda. Restart session, lalu Run all dari atas.')
        to_install.append(requirement)
if to_install:
    # Kendala ini mencegah pip mengganti pustaka numerik yang digunakan kernel.
    from pathlib import Path
    constraints = Path('/tmp/aes35_native_constraints.txt')
    constraints.write_text(''.join(p+'=='+metadata.version(p)+'\n' for p in CORE_REQUIREMENTS))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           '--constraint', str(constraints), *to_install])
print('Setup selesai. Semua cell berikutnya berjalan langsung di kernel notebook ini.')
print('Python:', sys.version.split()[0])


## 2 · Import dan ruang variabel notebook

In [ ]:
import os, sys, json, math, hashlib, random, warnings, re, itertools, unicodedata, zipfile, subprocess, gc, logging, html
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung PEFT.
    # Backend teks dikonfigurasi pada kernel notebook; tidak memuat operator vision.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from openpyxl import Workbook, load_workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
SCENARIOS = ('S1', 'S2', 'S3')
NOTEBOOK_CODE_ID = 'E17E_TPE_GPU_CONCAT_V1'
HPO_CODE_SHA256 = 'cb1ef8e1dcdb06e38bdd2f78b0e601b3532d1df265271af5912c1b52cb2a5574'
import optuna, sqlite3, shutil, tempfile, time, traceback
from optuna.trial import TrialState

from IPython.display import display, HTML


## 3 · Nama model pretrained

In [ ]:
# NAMA MODEL PRETRAINED — ubah di sini jika membuat eksperimen lain.
ROBERTA_CHECKPOINT = 'flax-community/indonesian-roberta-base'
BGE_CHECKPOINT = 'BAAI/bge-m3'

# Revision mengunci bobot dan tokenizer yang sama seperti E30.
ROBERTA_REVISION = 'cc5ecee7de8532880ad08d802292a6f42865bc83'
BGE_REVISION = '5617a9f61b028005a4858fdac845db406aefb181'

print('RoBERTa:', ROBERTA_CHECKPOINT, '| encoder ikut fine-tuning')
print('BGE-M3:', BGE_CHECKPOINT, '| encoder frozen; proyeksi/head tetap belajar')


## 4 · Pemetaan input dan empat target

In [ ]:
SCOPE = 'reset_e14_s1_s2_s3_multitrait'
SHEET_NAME = 'Sheet1'
QUESTION_COLUMN = 'Pertanyaan'
REFERENCE_COLUMN = 'Kunci_Jawaban'
ANSWER_COLUMN = 'Jawaban'
GROUP_COLUMN = 'Email Address'
RESPONSE_NUMBER_COLUMN = 'No_Soal'
INPUT_COLUMNS = (QUESTION_COLUMN, REFERENCE_COLUMN, ANSWER_COLUMN)

# Setel sesuai pedoman penilaian sebelum training. Rentang awal sesuai kategori di file contoh.
TRAITS = (
    ('isi', 'Skor_Isi', 1, 4),
    ('organisasi', 'Skor_Organisasi', 1, 2),
    ('kebahasaan', 'Skor_Tata_Bahasa', 1, 2),
    ('mekanik', 'Skor_Mekanik', 1, 2),
)
TARGET_COLUMNS = tuple(column for _, column, _, _ in TRAITS)
REQUIRED = (*INPUT_COLUMNS, GROUP_COLUMN, RESPONSE_NUMBER_COLUMN, *TARGET_COLUMNS)
RUBRIC = {'traits': [
    {'name': name, 'min': low, 'max': high, 'categories': list(range(low, high + 1))}
    for name, _, low, high in TRAITS
]}
SCORE_MIN = np.asarray([low for _, _, low, _ in TRAITS], dtype=np.float32)
SCORE_SPAN = np.asarray([high - low for _, _, low, high in TRAITS], dtype=np.float32)


## 5 · Kontrol E17C dan pengaturan pencarian

In [ ]:
# Resep E14 S3 dengan kandidat dropout MLP 0,1.
RUN_SCENARIOS = ('S3',)
if not RUN_SCENARIOS or set(RUN_SCENARIOS) - set(SCENARIOS) or len(set(RUN_SCENARIOS)) != len(RUN_SCENARIOS):
    raise ValueError('Pilih skenario unik dari S1, S2, dan S3.')
EPOCHS = 25
FOLDS = 5
EARLY_STOPPING_PATIENCE = 5
TRAIN_SEED = 42  # Satu konstanta pengacakan, tanpa loop multi-seed.
SPLIT_SEED = 2026
TEST_FRACTION = 0.20
MICRO_BATCH = 1
EFFECTIVE_BATCH = 8

ENCODER_LR = 5e-6            # Full fine-tuning RoBERTa.
S3_MLP_LR = 1e-4             # Head bersama.
S3_LAYERS_LR = 1e-4          # LayerNorm RoBERTa.
PROJECTION_LR = 1e-4         # Proyeksi dan LayerNorm BGE.

DROPOUT = 0.1               # Dropout encoder mengikuti checkpoint.
S3_MLP_DROPOUT = 0.1        # Kontrol E14 memakai 0,2.
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRADIENT_CLIP = 1.0

ROBERTA_MAX_TOKENS = 512
BGE_MAX_TOKENS = 1024
WINDOW_OVERLAP = 128
CONTEXT_MAX_TOKENS = 192
QUESTION_MAX_TOKENS = 64
DEVICE = 'cuda'



# Jalankan mode satu per satu. Mengubah STAGE tidak mengubah identitas study.
STAGE = 'confirm_final'             # 'confirm_final' menjalankan konfirmasi lalu final.
HPO_TRIALS = 60              # Anggaran COMPLETE + PRUNED; FAIL tidak dihitung.
TOP_K = 5
HPO_SEED = 2026
HPO_SPLIT_SEED = 2029
STABILITY_SEEDS = (42, 43, 44)
if STAGE not in ('search', 'confirm', 'final', 'confirm_final'):
    raise ValueError('STAGE tidak dikenal.')


### `config`

Periksa konfigurasi model, ukuran batch, dan anggaran token.

In [ ]:
# Nilai bawaan dan fungsi konfigurasi, tersedia langsung di notebook.

DEFAULTS = {
    'sheet': SHEET_NAME, 'expected_rows': None,
    'test_fraction': TEST_FRACTION, 'folds': FOLDS, 'seed': TRAIN_SEED, 'split_seed': SPLIT_SEED, 'split_candidates': 128,
    # Maksimal 25 epoch/fold; berhenti bila RMSE validasi tidak membaik selama patience.
    'epochs': EPOCHS, 'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'effective_batch': EFFECTIVE_BATCH, 'micro_batch': MICRO_BATCH,
    'encoder_lr': ENCODER_LR, 's3_encoder_lr': ENCODER_LR, 'projection_lr': PROJECTION_LR, 'head_lr': 1e-4, 'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO, 'gradient_clip': GRADIENT_CLIP, 'dropout': DROPOUT, 's3_mlp_dropout': S3_MLP_DROPOUT,
    'roberta_max_length': ROBERTA_MAX_TOKENS, 'roberta_overlap': WINDOW_OVERLAP,
    'bge_max_length': BGE_MAX_TOKENS, 'bge_overlap': WINDOW_OVERLAP, 'bge_batch': 4, 'device': DEVICE,
    'roberta': {'checkpoint': ROBERTA_CHECKPOINT,
                'revision': ROBERTA_REVISION},
    'bge': {'checkpoint': BGE_CHECKPOINT,
            'revision': BGE_REVISION},
    # Batas konteks dihitung termasuk prefix; semua token jawaban tetap dicakup.
    'context_max_tokens': CONTEXT_MAX_TOKENS, 'question_max_tokens': QUESTION_MAX_TOKENS,
    'new_question': '', 'new_reference': '',
    'input_policy':'three_texts_all_scenarios_v1', 'input_format':'single_sequence',
    's3_mlp_lr': S3_MLP_LR, 's3_layers_lr': S3_LAYERS_LR,
    's3_fusion': 'concat', 
    'vector_source': 'pretrained', 'trace_position': 0,
}


# Gabungkan parameter pilihan dengan nilai bawaan, lalu periksa batas skor, batch, dan panjang input.
def config(values):
    result = {**DEFAULTS, **values}
    if not 0 < result['test_fraction'] < 1 or result['folds'] < 2 or result['split_candidates'] < 1:
        raise ValueError('Split/folds tidak sah')
    if result['epochs'] < 1 or not 1 <= result['micro_batch'] <= result['effective_batch']:
        raise ValueError('Epoch/batch tidak sah')
    if type(result['early_stopping_patience']) is not int or result['early_stopping_patience'] < 1:
        raise ValueError('early_stopping_patience harus bilangan bulat positif.')
    if result['vector_source'] not in ('pretrained', 'pretrained_pair', 'S3'):
        raise ValueError('vector_source harus pretrained, pretrained_pair, atau S3')
    if result['device'] not in ('auto', 'cpu', 'cuda'):
        raise ValueError('device harus auto, cpu, atau cuda')
    if not 0 <= result['warmup_ratio'] < 1:
        raise ValueError('warmup_ratio tidak sah')
    if any(not math.isfinite(result[k]) or result[k] <= 0
           for k in ('encoder_lr', 's3_encoder_lr', 'projection_lr', 'head_lr', 's3_mlp_lr', 's3_layers_lr', 'gradient_clip', 'bge_batch')):
        raise ValueError('Learning rate, gradient clip, dan batch BGE harus positif.')
    if not 0 <= result['s3_mlp_dropout'] < 1:
        raise ValueError('s3_mlp_dropout harus berada pada rentang [0,1).')
    if not 0 <= result['dropout'] < 1 or result['weight_decay'] < 0:
        raise ValueError('Dropout/weight decay tidak sah.')
    if not 2 < result['roberta_max_length'] <= 512 or not 0 <= result['roberta_overlap'] < result['roberta_max_length']-2:
        raise ValueError('Window RoBERTa melebihi batas atau overlap tidak sah.')
    if not 2 < result['bge_max_length'] <= 8192:
        raise ValueError('Batas BGE harus 3–8192 token termasuk special token.')
    if not (type(result['context_max_tokens']) is int and type(result['question_max_tokens']) is int
            and 1 <= result['question_max_tokens'] < result['context_max_tokens']
            and result['context_max_tokens']+4+result['roberta_overlap'] < result['roberta_max_length']):
        raise ValueError('Anggaran konteks harus menyisakan token jawaban lebih banyak daripada overlap.')
    if str(result.get('new_answer', '')).strip() and not all(str(result.get(k, '')).strip() for k in ('new_question', 'new_reference')):
        raise ValueError('Untuk NEW_ANSWER baru, isi juga NEW_QUESTION dan NEW_REFERENCE.')
    if not (type(result['bge_overlap']) is int and 0 <= result['bge_overlap']
            < result['bge_max_length']-result['context_max_tokens']-4):
        raise ValueError('Anggaran konteks BGE harus menyisakan kapasitas jawaban di atas overlap.')
    if result['input_policy']!='three_texts_all_scenarios_v1':
        raise ValueError('Notebook ini khusus tiga input pada semua skenario.')
    if result['input_format'] not in ('single_sequence','context_pair'):
        raise ValueError('Format input tidak sah.')
    if result['s3_fusion'] != 'concat':
        raise ValueError('E17C memerlukan fusion concat.')
    return result


### `device_name`

Tentukan perangkat untuk encoder dan training.

In [ ]:
def device_name(cfg):
    requested = cfg['device']
    if requested == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA diminta tetapi tidak tersedia; pilih GPU di Colab atau device=cpu.')
    return ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested


### Periksa runtime

Tabel menunjukkan perangkat yang benar-benar tersedia di sesi ini.

In [ ]:
"""Native notebook runtime checks for data preparation and model execution."""


def runtime_information():
    available = torch.cuda.is_available()
    return {
        "torch": str(torch.__version__),
        "transformers": version("transformers"),
        "scikit-learn": version("scikit-learn"),
        "device": "cuda" if available else "cpu",
        "accelerator": torch.cuda.get_device_name(0) if available else None,
    }


def model_stage_ready(stage, required=()):
    """Report unavailable prerequisites before a model stage is executed."""
    missing_data = [
        name
        for name in ("cfg", "bundle", "manifest", "output")
        if name not in globals()
    ]
    if missing_data:
        print(
            f"{stage} belum dijalankan. Jalankan cell Siapkan dataset dan split "
            f'terlebih dahulu. Variabel yang diperlukan: {", ".join(missing_data)}.'
        )
        return False
    if cfg["device"] != "cpu" and not torch.cuda.is_available():
        print(
            f"{stage} belum dijalankan karena GPU CUDA belum tersedia. "
            "Dataset dan split sudah dapat diperiksa. Pilih Runtime → Change runtime type → T4 GPU, "
            "lalu Run all pada runtime GPU."
        )
        return False
    missing_stage = [name for name in required if name not in globals()]
    if missing_stage:
        print(
            f"{stage} belum dijalankan. Selesaikan tahap model sebelumnya. "
            f'Variabel yang diperlukan: {", ".join(missing_stage)}.'
        )
        return False
    return True


if not torch.cuda.is_available():
    raise RuntimeError('E17E khusus GPU. Pilih Runtime → Change runtime type → T4 GPU.')
runtime_status = runtime_information()
display(pd.DataFrame([runtime_status]))
if not torch.cuda.is_available():
    print(
        "Runtime CPU terdeteksi. Pembacaan workbook dan pembagian data tetap berjalan. "
        'Tahap model menunggu GPU. Untuk uji CPU yang disengaja, set DEVICE = "cpu" '
        "di hyperparameter lalu jalankan ulang konfigurasi dari atas."
    )


## 6 · Dataset semi-final di Google Drive

### `read_dataset` dan `Bundle`

Workbook dibaca sebagai tiga kolom teks dan matriks empat skor. Email hanya dipakai untuk membentuk kelompok split.

In [ ]:
# Baca dan audit data tanpa mengubah file Excel asli.
# Rapikan Unicode dan karakter kontrol; ejaan, huruf besar, dan tanda baca jawaban tetap dipertahankan.
def clean(text, normalize_punctuation=False):
    text = unicodedata.normalize('NFC', text).replace('\r\n','\n').replace('\r','\n')
    text = ''.join(c for c in text if c in '\n\t' or unicodedata.category(c) != 'Cc')
    if normalize_punctuation:
        text = text.translate(str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-'})).replace('…','...')
    return text

def digest(path):
    # Checkpoint besar diperiksa bertahap supaya tidak dimuat seluruhnya ke RAM.
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.pending')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    temporary.replace(path)


@dataclass
class Bundle:
    rows: list
    y: np.ndarray
    groups: np.ndarray
    source_sha256: str
    summary: dict

    @property
    def texts(self):
        return [r['text'] for r in self.rows]


# Baca jawaban dan nilai guru, periksa kelengkapan, lalu kelompokkan email/duplikat untuk mencegah kebocoran data.
def read_dataset(cfg):
    path = Path(cfg['dataset'])
    source_hash = digest(path)
    book = load_workbook(path, read_only=True, data_only=True)
    try:
        if cfg['sheet'] not in book.sheetnames:
            raise ValueError(f"Tab {cfg['sheet']} tidak tersedia: {book.sheetnames}")
        data = iter(book[cfg['sheet']].iter_rows(values_only=True))
        header = [str(x).strip() if x is not None else None for x in next(data, ())]
        if any(header.count(x) != 1 for x in REQUIRED):
            raise ValueError('Header wajib hilang atau duplikat: '+', '.join(REQUIRED))
        rows = []
        for excel_row, values in enumerate(data, 2):
            if all(v is None for v in values):
                continue  # Hanya baris format kosong, bukan label/jawaban yang hilang.
            raw = {k: values[i] for i, k in enumerate(header) if k is not None}
            missing = [k for k in REQUIRED if raw[k] is None or not str(raw[k]).strip()]
            if missing:
                raise ValueError(f'Baris {excel_row}: kolom wajib kosong: '+', '.join(missing))
            scores = []
            for _, column, low, high in TRAITS:
                score = raw[column]
                if (isinstance(score, bool) or not isinstance(score, (int, float))
                    or not math.isfinite(score) or int(score) != score or not low <= score <= high):
                    raise ValueError(f'Baris {excel_row}: {column} harus bilangan bulat {low}–{high}.')
                scores.append(int(score))
            for key in (QUESTION_COLUMN,REFERENCE_COLUMN):
                if not isinstance(raw[key],str) or not raw[key].strip():
                    raise ValueError(f'Baris {excel_row}: {key} harus teks yang tidak kosong.')
            if not isinstance(raw[ANSWER_COLUMN], str) or not clean(raw[ANSWER_COLUMN]).strip():
                raise ValueError(f'Baris {excel_row}: jawaban bukan teks.')
            rows.append({'row_id': f'row_{excel_row-1:04d}', 'source_row': excel_row,
                         'raw_text': raw[ANSWER_COLUMN], 'text': clean(raw[ANSWER_COLUMN]),
                         'email_group': unicodedata.normalize('NFC', str(raw[GROUP_COLUMN])).strip().casefold(),
                         'question': clean(raw[QUESTION_COLUMN]), 'reference': clean(raw[REFERENCE_COLUMN]),
                         'response_number': str(raw[RESPONSE_NUMBER_COLUMN]), 'scores': scores})
    finally:
        book.close()
    if cfg.get('expected_rows') is not None and len(rows) != cfg['expected_rows']:
        raise ValueError(f"Diharapkan {cfg['expected_rows']} baris, ditemukan {len(rows)}; tidak dipotong otomatis.")
    if not rows:
        raise ValueError('Dataset kosong')
    parent = list(range(len(rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        parent[find(b)] = find(a)

    email_first, duplicate_sets = {}, {}
    for i, row in enumerate(rows):
        email = row['email_group']
        if email in email_first:
            union(email_first[email], i)
        email_first[email] = i
        # Normalisasi ini HANYA audit kesamaan, bukan teks masukan encoder.
        normalized = ' '.join(row['text'].casefold().split())
        key = hashlib.sha256(normalized.encode()).hexdigest()
        duplicate_sets.setdefault(key, []).append(i)
    for indices in duplicate_sets.values():
        for i in indices[1:]:
            union(indices[0], i)
    roots = {root: i for i, root in enumerate(sorted({find(i) for i in range(len(rows))}))}
    groups = np.asarray([roots[find(i)] for i in range(len(rows))], dtype=int)
    y = np.asarray([row.pop('scores') for row in rows], dtype=np.float32)
    summary = {
        'scope': SCOPE, 'rows': len(rows), 'target_columns': list(TARGET_COLUMNS),
        'model_outputs': len(TRAITS), 'categories': RUBRIC['traits'],
        'email_groups': len(email_first), 'connected_groups': len(roots),
        'questions': len({row['response_number'] for row in rows}),
        'duplicate_sets': sum(len(indices) > 1 for indices in duplicate_sets.values()),
        'source_sha256': source_hash, 'input_features': list(INPUT_COLUMNS),
    }
    for row in rows:
        del row['email_group']
    return Bundle(rows, y, groups, source_hash, summary)

# Identitas hasil mengikuti data, konfigurasi, dan kode notebook ini.
def run_folder(cfg, bundle):
    code_hash = NOTEBOOK_CODE_ID
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    key = hashlib.sha256(json.dumps({'source': bundle.source_sha256, 'config': semantic, 'code': code_hash}, sort_keys=True).encode()).hexdigest()[:16]
    folder = Path(cfg['output']) / key
    folder.mkdir(parents=True, exist_ok=True)
    dump(folder / 'effective_config.json', cfg)
    return folder


### storage

Cell ini menjalankan bagian storage.

In [ ]:
# Notebook eksperimen berikutnya bisa memakai path dan SHA-256 yang sama.
DATASET_VERSION = 'semi-final'
DATASET_NAME = 'data_cleaned_3sekolah.xlsx'
DATASET_SHA256 = 'dfbd97128b09a4eb488dd890ccc3efd9b9fe232e74fcc00ee5504a03acc58226'
DRIVE_MOUNT = Path('/content/drive')
DRIVE_DATASET = DRIVE_MOUNT / 'MyDrive' / 'AES' / 'datasets' / 'semi_final' / DATASET_NAME
LOCAL_DATASET = Path(DATASET_NAME)  # Untuk eksekusi di luar Colab.
OUTPUT_ROOT = DRIVE_MOUNT / 'MyDrive' / 'AES' / 'outputs' / 'aes_reset_e17e_tpe_gpu'


def resolve_colab_dataset(mount_drive, upload, drive_mount, drive_dataset):
    """Gunakan salinan privat Drive atau simpan workbook yang cocok pada upload pertama."""
    from tempfile import NamedTemporaryFile

    mount_drive(str(drive_mount))
    if not (drive_mount / 'MyDrive').is_dir():
        raise RuntimeError('MyDrive belum tersedia. Hubungkan Google Drive di Colab.')
    if not drive_dataset.exists():
        print(f'Unggah {DATASET_NAME} untuk penyimpanan pertama di Drive.')
        uploads = upload()
        if len(uploads) != 1:
            raise ValueError(f'Unggah satu workbook asli bernama {DATASET_NAME}.')
        workbook = next(iter(uploads.values()))
        if hashlib.sha256(workbook).hexdigest() != DATASET_SHA256:
            raise ValueError('File unggahan berbeda dari workbook semi-final yang dipakai eksperimen ini.')
        drive_dataset.parent.mkdir(parents=True, exist_ok=True)
        with NamedTemporaryFile(dir=drive_dataset.parent, suffix='.pending', delete=False) as stream:
            staging = Path(stream.name)
            try:
                stream.write(workbook)
                stream.close()
                if digest(staging) != DATASET_SHA256:
                    raise IOError('Salinan di Drive gagal diverifikasi.')
                if not drive_dataset.exists():
                    staging.replace(drive_dataset)
            finally:
                staging.unlink(missing_ok=True)
        print(f'Workbook semi-final tersedia di Drive: {drive_dataset}')
    return drive_dataset


if 'google.colab' in sys.modules or Path('/content').is_dir():
    from google.colab import drive, files
    DATASET_PATH = resolve_colab_dataset(drive.mount, files.upload, DRIVE_MOUNT, DRIVE_DATASET)
else:
    DATASET_PATH = LOCAL_DATASET

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Workbook semi-final belum ditemukan: {DATASET_PATH}')
if digest(DATASET_PATH) != DATASET_SHA256:
    raise ValueError(f'Workbook di {DATASET_PATH} berbeda dari versi semi-final terpilih. '
                     'Gunakan file asli atau tetapkan path dan SHA-256 versi baru secara eksplisit.')
print(f'Dataset {DATASET_VERSION} terverifikasi: {DATASET_PATH}')


## 7 · Baca workbook dan periksa X serta y

### Definisi split siswa

Fungsi ini tersedia sebelum dataset dan manifest disiapkan.

In [ ]:
def make_split(bundle, cfg):
    indices = np.arange(len(bundle.rows))
    splitter = GroupShuffleSplit(n_splits=cfg['split_candidates'], test_size=cfg['test_fraction'], random_state=cfg['split_seed'])
    # Pilih kedekatan JUMLAH BARIS dengan target 80:20, bukan skor/hasil model.
    # GroupShuffleSplit membagi kelompok; ukuran kelompok tidak selalu sama.
    candidates = list(splitter.split(indices, groups=bundle.groups))
    development, test = min(candidates, key=lambda pair: abs(len(pair[1])/len(indices)-cfg['test_fraction']))
    if len(set(bundle.groups[development])) < cfg['folds']:
        raise ValueError('Kelompok training tidak cukup untuk CV yang diminta.')
    folds = []
    for number, (a, b) in enumerate(GroupKFold(n_splits=cfg['folds']).split(development, groups=bundle.groups[development]), 1):
        training, validation = development[a], development[b]
        assert not set(bundle.groups[training]) & set(bundle.groups[validation])
        folds.append({'fold': number, 'train': training.tolist(), 'validation': validation.tolist()})
    assert not set(bundle.groups[development]) & set(bundle.groups[test])
    manifest = {'scope': SCOPE, 'source_sha256': bundle.source_sha256,
                'development': development.tolist(), 'test': test.tolist(), 'folds': folds,
                'development_rows': len(development), 'test_rows': len(test),
                'development_groups': len(set(bundle.groups[development])),
                'test_groups': len(set(bundle.groups[test])),
                'target_test_row_fraction': cfg['test_fraction'],
                'candidate_group_fraction': cfg['test_fraction'],
                'split_candidates': cfg['split_candidates'],
                'selection_uses_labels': False,
                'actual_test_row_fraction': len(test)/len(indices),
                'policy': 'Holdout kelompok: pilih kandidat deterministik terdekat target proporsi baris berdasarkan ukuran kelompok saja, tanpa label/hasil model. CV hanya di development. Email/duplikat tidak menyeberang. Mengukur kelompok baru pada pertanyaan ini, bukan unseen-prompt.',
                'test_used_for_selection': False}
    manifest['split_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest


In [ ]:
cfg = config({
    'dataset': str(DATASET_PATH), 'output': str(OUTPUT_ROOT),
    'experiment': 'Reset_E17E_TPE_Concat', 'head_type': 'shared_mlp', 'trait_hidden': 256, 'series': 'multitrait_reset', 'dataset_version': DATASET_VERSION, 'traits': TRAITS,
    'runtime': runtime_information(),
    'fusion_variant': 'concat', 'trait_corr_threshold': 0.7, 'trait_relation_weight': 0.0,
})
bundle = read_dataset(cfg)
print(f"{len(bundle.rows)} jawaban, {bundle.summary['email_groups']} akun, "
      f"{bundle.summary['connected_groups']} kelompok setelah penggabungan duplikat.")
print(f"X berisi {len(INPUT_COLUMNS)} kolom teks: {', '.join(INPUT_COLUMNS)}")
print(f"y berukuran {bundle.y.shape}")
display(pd.DataFrame([{
    'trait': name, 'kolom_target': column, 'minimum': low, 'maksimum': high,
} for name, column, low, high in TRAITS]))

# Selesaikan dataset dan split pada cell yang sama.
manifest = make_split(bundle, cfg)
output = run_folder(cfg, bundle)
dump(output / 'split.json', manifest)
print(f"Split siap: {len(manifest['development'])} development, {len(manifest['test'])} test.")


## 8 · Pisahkan development, test, dan lima fold

### `contexts_from_bundle`

Ambil pasangan pertanyaan dan referensi yang mengiringi setiap jawaban.

In [ ]:
def contexts_from_bundle(bundle):
    """Ambil soal/referensi tanpa membaca target skor guru."""
    contexts = []
    for row in bundle.rows:
        question, reference = row.get('question'), row.get('reference')
        if not isinstance(question, str) or not question.strip():
            raise ValueError('Context pair memerlukan soal yang tidak kosong.')
        if not isinstance(reference, str) or not reference.strip():
            raise ValueError('Context pair memerlukan jawaban referensi yang tidak kosong.')
        contexts.append((clean(question), clean(reference)))
    return contexts


### split execute

Cell ini menjalankan bagian split execute.

In [ ]:
if all(name in globals() for name in ('cfg', 'bundle', 'manifest', 'output')):
    print(f"Development: {len(manifest['development'])} | Test: {len(manifest['test'])}")
    display(pd.DataFrame([{
        'fold': f['fold'], 'training': len(f['train']), 'validation': len(f['validation']),
    } for f in manifest['folds']]))
else:
    print('Jalankan cell Siapkan dataset dan split terlebih dahulu.')


## 9 · Tokenisasi, window, dan posisi input

### `mean_by_essay`

Rata-ratakan vektor window yang berasal dari jawaban yang sama.

In [ ]:
def mean_by_essay(cls, mapping, n):
    sums=cls.new_zeros((n,cls.shape[-1])).index_add(0,mapping,cls)
    counts=cls.new_zeros(n).index_add(0,mapping,cls.new_ones(len(mapping)))
    if (counts==0).any(): raise ValueError('Esai tanpa window')
    return sums / counts[:,None]


### `context_token_ids`

Ubah soal dan referensi jadi ID token sesuai batas konteks.

In [ ]:
def context_token_ids(context, tokenizer, cfg):
    """Sisakan anggaran untuk soal dan referensi sekaligus."""
    if not isinstance(context, (tuple, list)) or len(context) != 2:
        raise ValueError('Konteks harus pasangan (soal, jawaban referensi).')
    question, reference = context
    if not all(isinstance(t, str) and t.strip() for t in context):
        raise ValueError('Isi soal dan jawaban referensi sebelum memprediksi.')
    limit, qlimit = cfg['context_max_tokens'], cfg['question_max_tokens']
    if not 1 <= qlimit < limit:
        raise ValueError('Anggaran token soal harus lebih kecil daripada konteks.')
    # Prefix menjadi bagian input; nilai guru tidak pernah dimasukkan.
    qids = tokenizer.encode('Soal: '+question, add_special_tokens=False)
    rids = tokenizer.encode('\nJawaban referensi: '+reference, add_special_tokens=False)
    question_kept = qids[:qlimit]
    reference_kept = rids[:limit-len(question_kept)]
    return question_kept+reference_kept, {
        'question_tokens': len(qids), 'question_kept': len(question_kept),
        'reference_tokens': len(rids), 'reference_kept': len(reference_kept),
        'question_truncated': len(question_kept) < len(qids),
        'reference_truncated': len(reference_kept) < len(rids),
        'context_tokens_kept': len(question_kept)+len(reference_kept),
    }


### `reference_context_positions`

Tandai posisi token referensi untuk co-attention.

In [ ]:
def reference_context_positions(context, tok, cfg, info):
    """Posisi isi referensi dari offset karakter; ID token berulang tidak ditebak."""
    prefix = '\nJawaban referensi: '
    text = prefix + context[1]
    try:
        aligned = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    except NotImplementedError:
        # RoBERTa training tetap slow seperti E26; fast hanya membaca offset.
        if not hasattr(tok, '_aes_reference_fast'):
            spec = cfg['roberta']
            tok._aes_reference_fast = AutoTokenizer.from_pretrained(
                spec['checkpoint'], revision=spec['revision'], use_fast=True)
        aligned = tok._aes_reference_fast(text, add_special_tokens=False, return_offsets_mapping=True)
    original = tok.encode(text, add_special_tokens=False)
    if list(aligned['input_ids']) != original or len(aligned['offset_mapping']) != len(original):
        raise ValueError('ID/offset tokenizer referensi tidak identik; posisi tidak ditebak.')
    positions = []
    for index, (start, end) in enumerate(aligned['offset_mapping'][:info['reference_kept']]):
        if not 0 <= start <= end <= len(text):
            raise ValueError('Offset tokenizer referensi tidak sah.')
        # Token yang mencakup teks isi diperbolehkan; token prefix/spasi saja dikecualikan.
        if end > len(prefix) and text[max(start, len(prefix)):end].strip():
            positions.append(info['question_kept'] + index)
    if not positions:
        raise ValueError('Isi referensi habis oleh truncation/prefix. Tambah anggaran konteks sebelum training.')
    return positions


### `pair_content_windows`

Rangkai tiga teks dan bagi jawaban panjang menjadi window yang bertumpang tindih.

In [ ]:
def pair_content_windows(text, context, tokenizer, maximum=512, overlap=128, cfg=None):
    """Informasi dan window sama; E22 menambah separator resmi pasangan."""
    if cfg is None or cfg.get('input_format') not in ('single_sequence', 'context_pair'):
        raise ValueError('Pilih format input E21 atau E22 sebelum tokenisasi.')
    context_ids, context_info = context_token_ids(context, tokenizer, cfg)
    reference_positions = reference_context_positions(context, tokenizer, cfg, context_info)
    context_layout = [-1]*len(context_ids)
    for position in reference_positions:context_layout[position] = -4
    ids = tokenizer.encode(text, add_special_tokens=False)
    answer_prefix = tokenizer.encode('\nJawaban siswa:\n', add_special_tokens=False)
    if not ids:
        raise ValueError('Jawaban tidak menghasilkan token.')
    paired = cfg['input_format'] == 'context_pair'
    specials = tokenizer.num_special_tokens_to_add(pair=paired)
    # Cadangkan overhead terbesar untuk KEDUA eksperimen. Starts/cakupan sama.
    reserved = max(tokenizer.num_special_tokens_to_add(pair=False),
                   tokenizer.num_special_tokens_to_add(pair=True))
    capacity = maximum-len(context_ids)-len(answer_prefix)-reserved
    if not 0 <= overlap < capacity:
        raise ValueError('Konteks dan penanda jawaban menyisakan kapasitas di bawah overlap.')
    starts = [0]
    while starts[-1]+capacity < len(ids):
        starts.append(starts[-1]+capacity-overlap)
    chunks, answer_positions, reference_window_positions, segments = [], [], [], []
    for start in starts:
        answer = ids[start:start+capacity]
        second = answer_prefix+answer
        if paired:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids, second)
            layout = tokenizer.build_inputs_with_special_tokens(context_layout,
                     [-3]*len(answer_prefix)+[-2]*len(answer))
        else:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids+second)
            layout = tokenizer.build_inputs_with_special_tokens(
                     context_layout+[-3]*len(answer_prefix)+[-2]*len(answer))
        positions = [i for i, value in enumerate(layout) if value == -2]
        if (len(prepared) != len(layout) or len(prepared) > maximum
                or len(prepared)-len(context_ids)-len(second) != specials
                or [prepared[i] for i in positions] != answer):
            raise ValueError('Template tokenizer tidak konsisten dengan format input.')
        chunks.append({'input_ids':prepared, 'attention_mask':[1]*len(prepared)})
        answer_positions.append(positions)
        reference_window_positions.append([i for i, value in enumerate(layout) if value == -4])
        segments.append(['Soal/referensi' if v == -1 else 'Jawaban siswa' if v == -2
                         else 'Penanda jawaban' if v == -3 else 'Isi referensi' if v == -4 else 'Token khusus' for v in layout])
    stats = {**context_info, 'input_format':cfg['input_format'], 'content_tokens':len(ids),
             'capacity':capacity, 'stride':capacity-overlap, 'starts':starts,
             'windows':len(chunks), 'pair_special_tokens':specials,
             'reserved_special_tokens':reserved, 'answer_prefix_tokens':len(answer_prefix),
             'answer_positions':answer_positions, 'reference_positions':reference_window_positions,
             'reference_content_kept':len(reference_positions), 'segments':segments, 'answer_tokens_uncovered':0}
    return chunks, stats


### `pair_batch_windows`

Tambahkan padding dan simpan pemetaan window ke jawaban.

In [ ]:
def pair_batch_windows(texts, contexts, tokenizer, device='cpu', maximum=512, overlap=128, cfg=None):
    """Susun batch pasangan tanpa menukar konteks antarjawaban."""
    if len(texts) != len(contexts) or not texts:
        raise ValueError('Jumlah konteks harus sama dengan jumlah jawaban.')
    chunks, mapping, stats = [], [], []
    for index, (text, context) in enumerate(zip(texts, contexts)):
        values, detail = pair_content_windows(text, context, tokenizer, maximum, overlap, cfg)
        chunks.extend(values)
        mapping.extend([index]*len(values))
        stats.append(detail)
    batch = tokenizer.pad(chunks, padding=True, return_tensors='pt')
    return {key: value.to(device) for key, value in batch.items()}, torch.tensor(mapping, device=device), stats


## 10 · Memuat tokenizer dan encoder pretrained

### `release`

Bebaskan alokasi yang selesai dipakai sebelum berpindah tahap.

In [ ]:
def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


### ExpectedPoolerNotice

Cell ini menjalankan bagian ExpectedPoolerNotice.

In [ ]:
class ExpectedPoolerNotice(logging.Filter):
    """Hanya saring pemberitahuan dua bobot pooler yang memang tidak dipakai."""
    def filter(self, record):
        import ast
        import re
        message = record.getMessage()
        if not message.startswith('Some weights of the model checkpoint at ') or 'were not used when initializing' not in message:
            return True
        match = re.search(r"\['[^\n]+?\]", message)
        if match is None:
            return True
        try:
            keys = ast.literal_eval(match.group())
        except (ValueError, SyntaxError):
            return True
        return set(keys) != {'pooler.dense.bias', 'pooler.dense.weight'}


### `encoder`

Muat bobot pretrained. Jalur BGE dibekukan dan RoBERTa disiapkan untuk fine-tuning.

In [ ]:
def encoder(cfg, role):
    # Ambil nama checkpoint dan revision model yang dipilih: roberta atau bge.
    spec = cfg[role]
    logger = logging.getLogger('transformers.modeling_utils')
    notice = ExpectedPoolerNotice()
    logger.addFilter(notice)
    try:
        # Muat arsitektur dan bobot pretrained. Ini baru inisialisasi encoder, belum latihan memakai nilai guru.
        result = AutoModel.from_pretrained(spec['checkpoint'], revision=spec['revision'], add_pooling_layer=False)
    finally:
        logger.removeFilter(notice)
    # Periksa panjang vektor: RoBERTa 768 angka, BGE-M3 1024 angka.
    expected = 768 if role == 'roberta' else 1024
    if result.config.hidden_size != expected:
        raise ValueError(f'{role}: dimensi checkpoint tidak cocok dengan {expected}.')
    # Parameter RoBERTa ikut diperbarui; parameter BGE tidak ikut diperbarui.
    # RoBERTa boleh belajar; bobot encoder BGE tidak diperbarui.
    result.requires_grad_(role == 'roberta')
    return result


### `tokenizer`

Muat tokenizer dari repositori dan revisi yang cocok dengan encoder.

In [ ]:
def tokenizer(cfg, role, fast=False):
    # Muat tokenizer dari checkpoint yang sama agar ID token cocok dengan kosakata model.
    result = AutoTokenizer.from_pretrained(cfg[role]['checkpoint'], revision=cfg[role]['revision'],
                                           use_fast=fast if role == 'roberta' else True)
    if role == 'roberta' and not fast and result.num_special_tokens_to_add(False) != 2:
        raise ValueError('RoBERTa slow tokenizer harus menambahkan dua special token.')
    return result


### Cek window pada satu jawaban development

Tabel menampilkan panjang input dan jumlah token yang ditandai untuk referensi serta jawaban.

In [ ]:
if model_stage_ready('Cek tokenisasi', ()):
    # Ambil satu jawaban development dan perlihatkan ukuran window serta posisi input tiga teks untuk S3 concat.
    example_index = manifest['development'][0]
    example_row = bundle.rows[example_index]
    example_context = (example_row['question'], example_row['reference'])
    example_tokenizer = tokenizer(cfg, 'roberta')
    example_windows, example_detail = pair_content_windows(
        example_row['text'], example_context, example_tokenizer,
        cfg['roberta_max_length'], cfg['roberta_overlap'], cfg,
    )
    display(pd.DataFrame([{
        'id_jawaban': example_row['row_id'],
        'window': n,
        'panjang_input': len(window['input_ids']),
        'token_referensi': len(reference),
        'token_jawaban': len(answer),
    } for n, (window, reference, answer) in enumerate(zip(
        example_windows, example_detail['reference_positions'], example_detail['answer_positions'],
    ), 1)]))


## 11 · Fitur BGE-M3 frozen

### `paired_pretrained_values`

Gabungkan vektor dari semua window menjadi satu vektor per jawaban.

In [ ]:
def paired_pretrained_values(bundle, cfg, role):
    """Mean CLS antar-window; BGE dinormalisasi L2 setelah agregasi."""
    device = device_name(cfg)
    model, tok = encoder(cfg,role).to(device), tokenizer(cfg,role)
    model.eval()
    vectors, window_vectors, ids, coverage = [], [], [], []
    with torch.no_grad():
        for index, (text,context) in enumerate(zip(bundle.texts,contexts_from_bundle(bundle))):
            windows,mapping,details = pair_batch_windows([text],[context],tok,device,
                cfg[role+'_max_length'],cfg[role+'_overlap'],cfg)
            # Satu jawaban diproses bertahap agar jumlah window tidak meledakkan memori.
            cls_parts=[]
            for start in range(0,len(mapping),cfg['bge_batch']):
                part={k:v[start:start+cfg['bge_batch']] for k,v in windows.items()}
                cls_parts.append(model(**part).last_hidden_state[:,0,:].float())
            cls=torch.cat(cls_parts,dim=0)
            value=mean_by_essay(cls,mapping,1)
            if role=='bge':value=torch.nn.functional.normalize(value,dim=1)
            vectors.append(value.cpu().numpy())
            window_vectors.append(cls.cpu().numpy())
            row_id=bundle.rows[index].get('row_id',f'inference_{index}')
            ids.extend(f'{row_id}_window_{j}' for j in range(len(cls)))
            coverage.append({k:v for k,v in details[0].items() if k not in ('answer_positions','reference_positions','segments')})
            if (index+1)%25==0 or index+1==len(bundle.rows):
                print(f'{role}: tiga input {index+1}/{len(bundle.rows)} jawaban selesai.',flush=True)
    values=np.concatenate(vectors).astype(np.float32)
    if role=='bge' and any(p.requires_grad for p in model.parameters()):
        raise ValueError('Encoder BGE harus frozen.')
    del model,tok
    release()
    return values,np.concatenate(window_vectors),ids,coverage


### `bge_features`

Pakai cache bila teks dan konfigurasi tokenizer sama; vektor berdimensi 1024.

In [ ]:
def bge_features(bundle, cfg, output):
    """Cache berubah jika soal, referensi, jawaban, atau resep token berubah."""
    contexts=contexts_from_bundle(bundle)
    contract={'model':cfg['bge'], 'texts':bundle.texts, 'contexts':contexts,
              'maximum':cfg['bge_max_length'],'overlap':cfg['bge_overlap'],
              'context_max_tokens':cfg['context_max_tokens'],'question_max_tokens':cfg['question_max_tokens'],
              'policy':'three_inputs_mean_CLS_then_L2_v2', 'input_format':cfg['input_format']}
    key=hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:20]
    cache=Path(cfg['output'])/'cache'/f'bge_three_inputs_{key}.npz'
    metadata=cache.with_suffix('.json')
    if cache.exists() and metadata.exists():
        proof=json.loads(metadata.read_text())
        if proof['file_sha256']!=hashlib.sha256(cache.read_bytes()).hexdigest():
            raise ValueError('Cache BGE berubah; tidak dipakai.')
        with np.load(cache,allow_pickle=False) as data:values=data['features']
        reused=True
    else:
        values,_,_,coverage=paired_pretrained_values(bundle,cfg,'bge')
        cache.parent.mkdir(parents=True,exist_ok=True)
        np.savez_compressed(cache,features=values)
        proof={'model':cfg['bge'],'shape':list(values.shape),'frozen':True,
               'pooling':'mean paired-window CLS then L2','max_length':cfg['bge_max_length'],
               'feature_source':'Pertanyaan + Kunci_Jawaban + Jawaban', 'input_format':cfg['input_format'],
               'answer_tokens_uncovered':sum(s['answer_tokens_uncovered'] for s in coverage),
               'context_truncated_rows':sum(s['question_truncated'] or s['reference_truncated'] for s in coverage),
               'file_sha256':hashlib.sha256(cache.read_bytes()).hexdigest()}
        dump(metadata,proof)
        reused=False
    if values.shape!=(len(bundle.rows),1024) or not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values,axis=1),1,atol=1e-4):
        raise ValueError('Fitur BGE shape/normalisasi tidak sah.')
    dump(Path(output)/'bge_encoding.json',{**proof,'reused':reused})
    return torch.from_numpy(values)


### Hitung fitur BGE untuk seluruh jawaban

Cell ini menampilkan progres encoding dan ukuran matriks vektor yang akan dipakai pada CV dan refit.

In [ ]:
if model_stage_ready('Encoding BGE', ()):
    bge = bge_features(bundle, cfg, output)
    coverage = json.loads((output / 'bge_encoding.json').read_text())
    display(pd.DataFrame([{
        'encoder_frozen': cfg['bge']['checkpoint'],
        'jumlah_vektor': bge.shape[0], 'dimensi': bge.shape[1],
        'cache_dipakai_ulang': coverage['reused'],
        'token_jawaban_tidak_tercakup': coverage['answer_tokens_uncovered'],
    }]))


## 12 · Model S3 concat

RoBERTa full fine-tuning dan BGE-M3 frozen menghasilkan fitur. Proyeksi BGE, LayerNorm dan MLP bersama ikut dilatih.

S1 dan S2 memakai satu encoder. S3 memakai concat RoBERTa dan proyeksi BGE. Setiap skenario mempunyai MLP 256 dengan empat keluaran.

In [ ]:
def correlated_trait_pairs(labels, threshold):
    # Correlation uses only the training partition of the current fit.
    centered = labels.astype(np.float64) - labels.mean(axis=0)
    norm = np.linalg.norm(centered, axis=0)
    denominators = np.outer(norm, norm)
    corr = np.divide(centered.T @ centered, denominators,
                     out=np.zeros_like(denominators), where=denominators > 0)
    return np.nonzero(np.triu(corr >= threshold, k=1))


def relation_error(prediction, targets, pairs):
    left, right = pairs
    if left.numel() == 0:
        return prediction.new_zeros(())
    error = prediction - targets
    return (error[:, left] - error[:, right]).square().mean()


class MultiTraitModel(nn.Module):
    def __init__(self, scenario, roberta, dropout=0.1):
        super().__init__()
        if scenario not in SCENARIOS:
            raise ValueError(f'Skenario tidak dikenal: {scenario}')
        self.scenario, self.roberta = scenario, roberta
        if self.roberta is not None:
            self.roberta.requires_grad_(True)
        if scenario == 'S3':
            self.roberta_norm = nn.LayerNorm(768)
            self.projection = nn.Sequential(nn.Linear(1024, 256), nn.LayerNorm(256))
        self.head = nn.Sequential(
            nn.Linear(768 if scenario == 'S1' else 1024, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, len(TRAITS)),
        )

    def forward(self, windows, mapping, n_essays, bge, return_features=False):
        features = {}
        if self.roberta is not None:
            hidden = self.roberta(**windows).last_hidden_state
            cls = mean_by_essay(hidden[:, 0, :], mapping, n_essays)
            features['roberta'] = cls
        if self.scenario == 'S1':
            representation = cls
        elif self.scenario == 'S2':
            representation = bge
        else:
            projected = self.projection(bge)
            representation = torch.cat([self.roberta_norm(cls), projected], dim=1)
            features.update(bge=bge, projection=projected, concat=representation)
        prediction = self.head(representation)
        return (prediction, features) if return_features else prediction


### `Engine`

Siapkan input batch dan kembalikan prediksi dari skala normalisasi ke poin tiap trait.

In [ ]:
class Engine:
    """Bangun input hanya untuk cabang encoder yang dipakai."""
    def __init__(self, model, texts, bge, cfg, device, roberta_tokenizer=None, contexts=None):
        self.model, self.texts, self.bge, self.cfg, self.device = model, texts, bge, cfg, device
        self.contexts = contexts
        if contexts is None or len(contexts) != len(texts):
            raise ValueError('Setiap jawaban memerlukan satu pasangan soal dan referensi.')
        self.tokenizer = (roberta_tokenizer or tokenizer(cfg, 'roberta')) if model.scenario != 'S2' else None

    def forward(self, indices, return_features=False):
        ids = [int(i) for i in indices]
        features = self.bge[ids].to(self.device) if self.model.scenario != 'S1' else None
        windows = mapping = None
        if self.model.scenario != 'S2':
            windows, mapping, _ = pair_batch_windows(
                [self.texts[i] for i in ids], [self.contexts[i] for i in ids],
                self.tokenizer, self.device, self.cfg['roberta_max_length'],
                self.cfg['roberta_overlap'], self.cfg)
        return self.model(windows, mapping, len(ids), features, return_features)

    def predict(self, indices):
        # Mode prediksi: nonaktifkan dropout. Pembacaan di bawah memakai no_grad.
        self.model.eval()
        values = []
        # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
        with torch.no_grad():
            for start in range(0, len(indices), self.cfg['micro_batch']):
                ids = indices[start:start+self.cfg['micro_batch']]
                values.append(self.forward(ids).float().cpu().numpy())
        scaled = np.concatenate(values)
        # Kembalikan setiap keluaran ke satuan poin trait.
        return scaled * SCORE_SPAN + SCORE_MIN


## 13 · Inisialisasi model dan spesifikasi

### `seed_everything`

Atur pengacakan training pada awal setiap fit mengikuti seed baseline sederhana.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### `make_model`

Buat model baru untuk setiap fold dan refit.

In [ ]:
def make_model(scenario, cfg):
    roberta = encoder(cfg, 'roberta') if scenario in ('S1', 'S3') else None
    return MultiTraitModel(scenario, roberta, dropout=cfg['s3_mlp_dropout'])


### architecture

Cell ini menjalankan bagian architecture.

In [ ]:
display(pd.DataFrame([
    ('S3', 'Concat RoBERTa 768 + proyeksi BGE 256', 1024, 256, S3_MLP_DROPOUT, 4),
], columns=['skenario', 'representasi', 'input_MLP', 'hidden_MLP', 'dropout_MLP', 'output']))

## 14 · Metrik evaluasi

RMSE dan Pearson memakai prediksi mentah. QWK dan Gwet AC2 memakai kategori terdekat.

In [ ]:
import warnings
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score

def ordinal_prediction(values, trait):
    cats=np.asarray(trait['categories'],dtype=float)
    raw=np.asarray(values,dtype=float)
    if not np.isfinite(raw).all():raise ValueError('Prediksi nonfinite')
    a=np.clip(raw,trait['min'],trait['max'])
    # argmin memilih kategori pertama (lebih rendah) ketika jarak sama.
    return cats[np.abs(a[...,None]-cats).argmin(-1)].astype(int)

def gwet_ac2(teacher, predicted, categories):
    a=np.asarray(teacher);b=np.asarray(predicted);c=np.asarray(categories,dtype=float)
    if len(a)!=len(b) or len(a)<2: raise ValueError('AC2 perlu minimal dua pasangan')
    if not np.isin(a,c).all() or not np.isin(b,c).all(): raise ValueError('Kategori AC2 tidak sah')
    q=len(c); weights=1-((c[:,None]-c[None,:])/(c.max()-c.min()))**2
    ai=np.searchsorted(c,a); bi=np.searchsorted(c,b)
    pa=weights[ai,bi].mean()
    pi=(np.bincount(ai,minlength=q)+np.bincount(bi,minlength=q))/(2*len(a))
    pe=weights.sum()*np.sum(pi*(1-pi))/(q*(q-1))
    return float((pa-pe)/(1-pe))

def evaluate(teacher, predicted, rubric):
    y=np.asarray(teacher,dtype=float);p=np.asarray(predicted,dtype=float)
    if y.shape!=p.shape or y.ndim!=2 or y.shape[1]!=4: raise ValueError('Metrik memerlukan [N,4] yang sama')
    if not np.isfinite(y).all() or not np.isfinite(p).all(): raise ValueError('Label/prediksi nonfinite')
    result={}
    for j,t in enumerate(rubric['traits']):
        a,b=y[:,j],p[:,j]; cats=t['categories']
        if not np.isin(a,cats).all(): raise ValueError('Label harus kategori guru yang sah')
        o=ordinal_prediction(b,t); reasons={}
        rmse=float(np.sqrt(np.mean((a-b)**2))) if len(a) else None
        pearson=None
        if len(a)>=2 and np.ptp(a)>0 and np.ptp(b)>0: pearson=float(pearsonr(a,b).statistic)
        else: reasons['Pearson']='Minimal dua pasangan dan kedua input tidak konstan.'
        qwk=None;ac2=None
        if len(a)>=2:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore'); v=cohen_kappa_score(a,o,weights='quadratic',labels=cats)
            if np.isfinite(v): qwk=float(v)
            else: reasons['QWK']='Expected disagreement nol; koefisien undefined.'
            ac2=gwet_ac2(a,o,cats)
        else: reasons.update({'QWK':'Minimal dua pasangan.','AC2':'Minimal dua pasangan.'})
        result[t['name']]={'RMSE':rmse,'Pearson':pearson,'QWK':qwk,'AC2':ac2,
             'out_of_range_fraction':float(((b<t['min']) | (b>t['max'])).mean()) if len(b) else None,
             'undefined_reasons':reasons}
    return result


### `metrics` dan `predictions_array`

Pemilihan epoch memakai akar rata-rata kuadrat error seluruh pasangan jawaban dan trait pada skala rubrik asli. Empat metrik per trait diekspor untuk analisis.

In [ ]:
def metrics(teacher, prediction):
    """RMSE validasi pada seluruh pasangan skor asli, serta metrik per trait."""
    per_trait = evaluate(teacher, prediction, RUBRIC)
    residual = np.asarray(teacher, dtype=float) - np.asarray(prediction, dtype=float)
    return {'RMSE': float(np.sqrt(np.mean(residual ** 2))), 'traits': per_trait}


def predictions_array(values, indices):
    array = np.asarray(values, dtype=float)
    if array.shape != (len(indices), len(TRAITS)) or not np.isfinite(array).all():
        raise ValueError('Prediksi harus finite dan berukuran [jumlah jawaban, 4].')
    return array


## 15 · Learnable Code — optimizer, loss, dan satu fit

### `optimizer_groups`

Kelompokkan parameter dengan LR RoBERTa 5e-6 dan LR komponen lain 1e-4.

In [ ]:
def optimizer_groups(model, cfg):
    """Kelompok optimizer per skenario; semua parameter trainable tercakup tepat sekali."""
    buckets = {'roberta': [], 'projection': [], 'head': [], 'pooling_norm': []}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith('roberta.'):
            group = 'roberta'
        elif name.startswith('projection.'):
            group = 'projection'
        elif name.startswith('head.'):
            group = 'head'
        else:
            group = 'pooling_norm'
        buckets[group].append(parameter)
    rates = {'roberta': cfg['s3_encoder_lr'] if model.scenario == 'S3' else cfg['encoder_lr'],
             'projection': cfg['projection_lr'],
             'head': cfg['s3_mlp_lr'] if model.scenario == 'S3' else cfg['head_lr'],
             'pooling_norm': cfg['s3_layers_lr']}
    result = [{'name': name, 'params': params, 'lr': rates[name]} for name, params in buckets.items() if params]
    ids = [id(p) for group in result for p in group['params']]
    expected = {id(p) for p in model.parameters() if p.requires_grad}
    if len(ids) != len(set(ids)) or set(ids) != expected:
        raise ValueError('Optimizer harus mencakup setiap parameter trainable tepat sekali.')
    return result


### `fit`

Latih satu model dengan AdamW, MSE target skala 0–1, serta pencatatan loss relasi antartrait. Saat CV, pilih epoch dengan RMSE validasi terkecil.

In [ ]:
def fit(scenario, training, validation, bundle, bge, cfg, epochs, seed, epoch_callback=None):
    pair_indices = correlated_trait_pairs(bundle.y[training], cfg['trait_corr_threshold'])
    active_pairs = [[TRAITS[i][0], TRAITS[j][0]] for i, j in zip(*pair_indices)]
    seed_everything(seed)
    device = device_name(cfg)
    pairs = tuple(torch.as_tensor(indices, device=device) for indices in pair_indices)
    print(f'Pasangan trait aktif: {active_pairs}', flush=True)
    # Buat model baru untuk fold ini, lalu tempatkan di GPU atau CPU yang tersedia.
    model = make_model(scenario, cfg).to(device)
    engine = Engine(model, bundle.texts, bge, cfg, device, contexts=contexts_from_bundle(bundle))
    groups = optimizer_groups(model, cfg)
    learning_rates = {group['name']: group['lr'] for group in groups}
    # AdamW memperbarui kelompok parameter sesuai learning rate masing-masing.
    optimizer = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader([int(i) for i in training], batch_size=cfg['effective_batch'],
                        shuffle=True, generator=generator, num_workers=0)
    total_steps = len(loader)*epochs
    warmup = min(int(cfg['warmup_ratio']*total_steps), max(0, total_steps-1))
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_steps)
    use_cuda = device == 'cuda'
    dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda and dtype == torch.float16)
    selected_parameters = {}
    for name, parameter in model.named_parameters():
        group = name.split('.')[0]
        if parameter.requires_grad and group not in selected_parameters:
            selected_parameters[group] = (name, parameter.detach().reshape(-1)[:32].cpu().clone())
    best_state, best_epoch, best_rmse = None, 0, float('inf')
    patience = cfg['early_stopping_patience']
    without_improvement, early_stopped = 0, False
    history, active_steps, optimizer_steps, skipped_steps = [], 0, 0, 0
    pruned = False
    for epoch in range(1, epochs+1):
        model.train()
        loss_total = mse_total = relation_total = 0.
        for effective in loader:
            batch = effective.tolist()
            optimizer.zero_grad(set_to_none=True)
            for start in range(0, len(batch), cfg['micro_batch']):
                ids = batch[start:start+cfg['micro_batch']]
                # Normalisasi per trait memberi bobot sama pada empat dimensi loss.
                targets = torch.as_tensor((bundle.y[ids] - SCORE_MIN) / SCORE_SPAN,
                                          dtype=torch.float32, device=device)
                with torch.autocast(device_type=device, dtype=dtype, enabled=use_cuda):
                    prediction = engine.forward(ids).float()
                    if prediction.shape != targets.shape:
                        raise ValueError('Shape prediksi/target harus [B,4], tanpa broadcasting.')
                    # MSE mengukur selisih kuadrat antara prediksi dan target; optimizer berusaha memperkecilnya.
                    mse = torch.mean((prediction-targets)**2)
                    relation = relation_error(prediction, targets, pairs)
                    loss = mse + cfg['trait_relation_weight'] * relation
                if not torch.isfinite(loss):
                    raise RuntimeError('Loss nonfinite; run dihentikan, tidak diklaim berhasil.')
                # Hitung gradien dan bobot microbatch sesuai ukuran batch efektif aktual.
                scaler.scale(loss*(len(ids)/len(batch))).backward()
                loss_total += loss.item()*len(ids)
                mse_total += mse.item()*len(ids)
                relation_total += relation.item()*len(ids)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
            lr_active = any(g['lr'] > 0 for g in optimizer.param_groups)
            previous_scale = scaler.get_scale()
            # Perbarui bobot; GradScaler bisa melewati langkah yang overflow pada AMP.
            scaler.step(optimizer)
            scaler.update()
            skipped = scaler.is_enabled() and scaler.get_scale() < previous_scale
            if skipped:
                skipped_steps += 1
            else:
                active_steps += int(lr_active)
                scheduler.step()
                optimizer_steps += 1
        row = {'scheduler_horizon_epochs': epochs, 'scheduler_total_steps': total_steps,
               'scheduler_warmup_steps': warmup, 'scheduler_step': optimizer_steps,
               'lr_end_roberta': optimizer.param_groups[0]['lr'], 'epoch': epoch, 'train_mse_scaled': mse_total/len(training),
               'train_relation_loss': relation_total/len(training),
               'train_total_loss': loss_total/len(training), 'active_trait_pairs': active_pairs,
               'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
               'effective_batch_target': cfg['effective_batch'], 'micro_batch': cfg['micro_batch']}
        if validation is not None:
            predicted = engine.predict(validation)
            row['validation'] = metrics(bundle.y[validation], predicted)
            # Simpan epoch dengan RMSE validasi terkecil; data uji akhir tidak ikut memilih epoch.
            if row['validation']['RMSE'] < best_rmse:
                best_rmse, best_epoch = row['validation']['RMSE'], epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                without_improvement = 0
            else:
                without_improvement += 1
            row.update(best_epoch_so_far=best_epoch, epochs_without_improvement=without_improvement)
            message = (f" | RMSE validasi: {row['validation']['RMSE']:.4f} poin"
                       f" | terbaik: epoch {best_epoch}, {best_rmse:.4f} poin"
                       f" | tanpa perbaikan: {without_improvement}/{patience}")
        else:
            best_epoch = epoch
            message = ' | pelatihan final; data uji belum dibuka'
        history.append(row)
        if epoch_callback is not None and epoch_callback(epoch, row, history):
            pruned = True
            break
        print(f'{scenario} · epoch {epoch}/{epochs} · total loss latih: {row["train_total_loss"]:.6f}'+message, flush=True)
        # Hanya fold validasi yang boleh menghentikan latihan; data uji tidak dibaca.
        if validation is not None and without_improvement >= patience and epoch < epochs:
            early_stopped = True
        if early_stopped:
            print(f'{scenario}: early stopping setelah {epoch} epoch; RMSE validasi tidak membaik selama {patience} epoch. '
                  f'Bobot epoch {best_epoch} dipulihkan.', flush=True)
            break
    if active_steps == 0:
        raise RuntimeError('Tidak ada pembaruan dengan LR aktif.')
    named = dict(model.named_parameters())
    changed = {group: not torch.equal(before, named[name].detach().reshape(-1)[:32].cpu())
               for group, (name, before) in selected_parameters.items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    info = {'pruned': pruned, 'scenario': scenario, 'best_epoch': best_epoch, 'epochs_completed': len(history),
            'max_epochs': epochs, 'early_stopping_patience': patience if validation is not None else None,
            'early_stopped': early_stopped, 'optimizer_learning_rates': learning_rates,
            'best_validation_rmse': best_rmse if validation is not None else None,
            'stop_reason': ('RMSE validasi tidak membaik selama batas kesabaran' if early_stopped else
                            'Batas maksimum epoch tercapai' if validation is not None else
                            'Jumlah epoch final yang dipilih dari CV selesai'),
            'history': history, 'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
            'parameter_samples_changed': changed, 'BGE_encoder_trained': False,
            'selection': 'minimum validation RMSE in original score units' if validation is not None else 'fixed refit epochs selected from CV'}
    # Bobot diambil SESUDAH best_state dipulihkan; sesuai model yang dievaluasi.
    return model, engine, info


## 16 · Validasi silang pada development

### `run_cv`

Latih S3 pada lima fold development. Setiap jawaban mendapatkan satu prediksi OOF dari model yang tidak dilatih memakai jawaban tersebut.

In [ ]:
def run_cv(bundle, manifest, bge, cfg, output, scenario):
    """Validasi silang lima fold dengan resep dan seed E13."""
    results = []
    for fold in manifest['folds']:
        number, validation = fold['fold'], fold['validation']
        path = output / f'fold_{number}.json'
        identity = {'split': manifest['split_sha256'], 'fold': number, 'validation': validation, 'scenario': scenario, 'config_sha256': fit_config_hash(cfg)}
        if path.exists():
            result = json.loads(path.read_text())
            if result['identity'] != identity:
                raise ValueError(f'Cache fold memakai split berbeda: {path}')
            print(f'E17E {scenario} · fold {number}: hasil selesai dipakai ulang.', flush=True)
        else:
            print(f'E17E {scenario} · fold {number}/{cfg["folds"]}', flush=True)
            model, engine, info = fit(scenario, fold['train'], validation, bundle, bge,
                                      cfg, cfg['epochs'], cfg['seed'] + number)
            prediction = predictions_array(engine.predict(validation), validation)
            result = {'identity': identity, 'prediction': prediction.tolist(), 'training': info}
            dump(path, result)
            del model, engine
            release()
        prediction = predictions_array(result['prediction'], validation)
        evaluation = metrics(bundle.y[validation], prediction)
        if not math.isclose(evaluation['RMSE'],
                            result['training']['best_validation_rmse'], abs_tol=1e-6):
            raise ValueError(f'Prediksi dan epoch terpilih tidak cocok: {path}')
        results.append({**result, 'metrics': evaluation})
    indices = [i for result in results for i in result['identity']['validation']]
    if sorted(indices) != sorted(manifest['development']):
        raise ValueError('Setiap jawaban development harus memiliki tepat satu prediksi OOF.')
    return {
        'eksperimen': 'Reset_E17E_TPE_Concat', 'mean_cv_RMSE': float(np.mean([
            r['metrics']['RMSE'] for r in results
        ])),
        'epoch_refit': math.ceil(np.median([r['training']['best_epoch'] for r in results])),
    }, results


## 17 · Penyimpanan hasil dan metrik
Excel memakai struktur laporan E17C; tabel HPO ditambahkan ke workbook05. Semua file hasil tetap privat.

In [ ]:
def save_model(path, model, cfg, bundle, manifest, info):
    payload = {'scope': SCOPE, 'trained': True, 'scenario': model.scenario,
               'source_sha256': bundle.source_sha256, 'split_sha256': manifest['split_sha256'],
               'config': cfg, 'training': info,
               'state': {k: v.detach().cpu() for k, v in model.state_dict().items()}}
    torch.save(payload, path)


In [ ]:
def refit_checkpoint(epochs, bundle, manifest, bge, cfg, output, scenario):
    """Latih pada development dan simpan checkpoint sebelum cell test dijalankan."""
    identity = {'refit_epochs': epochs, 'split': manifest['split_sha256'], 'scenario': scenario, 'config_sha256': fit_config_hash(cfg)}
    model_path = output / 'final.pt'
    refit_path = output / 'refit_result.json'
    test_path = output / 'test_result.json'
    # Format lama menyimpan informasi refit bersama hasil test.
    for saved in (refit_path, test_path):
        if saved.is_file():
            result = json.loads(saved.read_text())
            if (result['identity'] != identity or not model_path.is_file()
                or digest(model_path) != result['model_sha256']):
                raise ValueError('Metadata refit dan checkpoint final berbeda.')
            print('Checkpoint refit yang sudah selesai dipakai ulang.', flush=True)
            return result

    print(f'Refit E17E {scenario} · {epochs} epoch pada seluruh development.', flush=True)
    model, engine, info = fit(scenario, manifest['development'], None, bundle, bge, cfg,
                              epochs, cfg['seed'])
    pending = model_path.with_suffix('.pending.pt')
    save_model(pending, model, cfg, bundle, manifest, info)
    pending.replace(model_path)
    result = {'identity': identity, 'training': info, 'model_sha256': digest(model_path)}
    dump(refit_path, result)
    del model, engine
    release()
    return result


In [ ]:
def evaluate_test(refit_result, bundle, manifest, bge, cfg, output):
    """Baca checkpoint refit dan hitung satu prediksi untuk setiap jawaban test."""
    complete = output / 'test_result.json'
    model_path = output / 'final.pt'
    if digest(model_path) != refit_result['model_sha256']:
        raise ValueError('Checkpoint yang akan dievaluasi berbeda dari checkpoint refit.')
    test = manifest['test']
    if complete.exists():
        result = json.loads(complete.read_text())
        if (result['identity'] != refit_result['identity']
            or result['model_sha256'] != refit_result['model_sha256']):
            raise ValueError('Hasil test dan checkpoint final berbeda.')
        print('Hasil test final yang sudah selesai dipakai ulang.', flush=True)
        return predictions_array(result['prediction'], test)

    payload = torch.load(model_path, map_location='cpu', weights_only=True)
    device = device_name(cfg)
    model = make_model(refit_result['identity']['scenario'], cfg)
    model.load_state_dict(payload['state'], strict=True)
    del payload
    model.to(device)
    engine = Engine(model, bundle.texts, bge, cfg, device, contexts=contexts_from_bundle(bundle))
    prediction = predictions_array(engine.predict(test), test)
    dump(complete, {**refit_result, 'prediction': prediction.tolist()})
    del model, engine
    release()
    return prediction


In [ ]:
def excel_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, default=str)
    if isinstance(value, float) and not math.isfinite(value):
        return None  # Tidak tercakup/undefined: tetap kosong, bukan nol.
    if value is pd.NA:
        return None
    return value


In [ ]:
def write_excel(path, sheets, metadata):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    book = Workbook(write_only=True)
    used = set()
    counts = {}
    all_sheets = {'Info': pd.DataFrame([{'field': k, 'value': v} for k, v in metadata.items()]), **sheets}
    for name, frame in all_sheets.items():
        frame = frame if isinstance(frame, pd.DataFrame) else pd.DataFrame(frame)
        if len(frame.columns) > 16384:
            raise ValueError('Terlalu banyak kolom untuk Excel; tidak dipotong diam-diam.')
        headers = [str(x) for x in frame.columns]
        chunks = max(1, math.ceil(len(frame)/1048575))
        for part in range(chunks):
            base = re.sub(r'[\\/*?:\[\]]', '_', name)
            suffix = f'_{part+1}' if chunks > 1 else ''
            title = base[:31-len(suffix)]+suffix
            if title in used:
                raise ValueError('Nama sheet bertabrakan setelah dipendekkan: '+title)
            used.add(title)
            sheet = book.create_sheet(title)
            sheet.freeze_panes = 'B2'
            heading = []
            for text in headers:
                cell = WriteOnlyCell(sheet, value=text)
                cell.font = Font(bold=True, color='FFFFFF')
                cell.fill = PatternFill('solid', fgColor='172035')
                cell.data_type = 's'
                heading.append(cell)
            sheet.append(heading)
            subset = frame.iloc[part*1048575:(part+1)*1048575]
            for row in subset.itertuples(index=False, name=None):
                values = []
                for value in row:
                    value = excel_value(value)
                    if isinstance(value, str):
                        if len(value) > 32767:
                            raise ValueError('Teks melebihi batas satu sel Excel; ekspor dihentikan agar tidak terpotong.')
                        cell = WriteOnlyCell(sheet, value=value)
                        cell.data_type = 's'  # Teks siswa '=...' tidak dijalankan sebagai formula.
                        values.append(cell)
                    else:
                        values.append(value)
                sheet.append(values)
            counts[title] = {'rows': len(subset), 'columns': len(headers)}
    temporary = path.with_name(path.stem+'.pending.xlsx')
    book.save(temporary)
    temporary.replace(path)
    return counts


def export_excel_zip(path, groups, metadata, max_bytes=30_000_000):
    """Simpan workbook utuh per bagian. Pecah lagi jika ukuran aktual melewati batas."""
    from tempfile import TemporaryDirectory

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    inventory = []
    with TemporaryDirectory(prefix='xlsx_', dir=path.parent) as temporary:
        staging = Path(temporary)

        def write_part(stem, tables, relative_part='', row_start=0):
            candidate = staging / f'{stem}{relative_part}.xlsx'
            counts = write_excel(candidate, tables, metadata)
            size = candidate.stat().st_size
            if size < max_bytes:
                inventory.append({
                    'file': candidate.name, 'bytes': size, 'sha256': digest(candidate),
                    'sheet_sizes': counts,
                    'tables': [
                        {'name': name, 'row_start': row_start, 'rows': len(frame)}
                        for name, frame in tables.items()
                    ],
                })
                return

            candidate.unlink()
            if len(tables) > 1:
                entries = list(tables.items())
                middle = len(entries) // 2
                write_part(stem, dict(entries[:middle]), relative_part + '_1', row_start)
                write_part(stem, dict(entries[middle:]), relative_part + '_2', row_start)
                return

            name, frame = next(iter(tables.items()))
            if len(frame) <= 1:
                raise ValueError(f'Sheet {name} tidak dapat dipisah lagi agar di bawah {max_bytes:,} byte.')
            middle = len(frame) // 2
            write_part(stem, {name: frame.iloc[:middle]}, relative_part + '_1', row_start)
            write_part(stem, {name: frame.iloc[middle:]}, relative_part + '_2', row_start + middle)

        for stem, tables in groups.items():
            if tables:
                write_part(stem, tables)

        # XLSX sudah dikompresi. Simpan langsung tanpa mengulang kompresi ZIP.
        pending = staging / 'hasil.zip'
        folder = path.stem
        manifest_name = f'{folder}/manifest.json'
        with zipfile.ZipFile(pending, 'w', compression=zipfile.ZIP_STORED) as archive:
            for entry in inventory:
                archive.write(staging / entry['file'], f"{folder}/{entry['file']}")
            archive.writestr(manifest_name, json.dumps({
                'xlsx_size_limit_bytes_exclusive': max_bytes,
                'table_row_start_is_zero_based': True,
                'workbooks': inventory,
            }, ensure_ascii=False, indent=2))

        with zipfile.ZipFile(pending) as archive:
            sizes = {item.filename: item.file_size for item in archive.infolist()}
            expected = {f"{folder}/{entry['file']}": entry['bytes'] for entry in inventory}
            if (set(sizes) != {*expected, manifest_name}
                or any(sizes[name] != size or size >= max_bytes for name, size in expected.items())
                or archive.testzip() is not None):
                raise IOError('Isi ZIP gagal diverifikasi.')
        pending.replace(path)
    return inventory


In [ ]:
def lowest_score_rows(bundle, indices, prediction, stage):
    scores = predictions_array(prediction, indices)
    rows = []
    for column, trait in enumerate(RUBRIC['traits']):
        predicted = ordinal_prediction(scores[:, column], trait)
        lowest = bundle.y[indices, column] == trait['min']
        total = int(lowest.sum())
        correct = int(np.sum(lowest & (predicted == trait['min'])))
        rows.append({
            'bagian': stage, 'trait': trait['name'], 'skor_terendah': trait['min'],
            'n_label_terendah': total, 'n_terdeteksi': correct,
            'proporsi_terdeteksi': correct / total if total else None,
        })
    return rows


def trait_rows(values, **keys):
    return [{**keys, 'trait': name,
             'RMSE': stats['RMSE'], 'Pearson': stats['Pearson'],
             'QWK': stats['QWK'], 'Gwet AC2': stats['AC2'],
             'proporsi_di_luar_skala': stats['out_of_range_fraction'],
             'alasan_metrik_kosong': stats['undefined_reasons']}
            for name, stats in values['traits'].items()]


def prediction_rows(bundle, indices, prediction, **keys):
    scores = predictions_array(prediction, indices)
    categories = np.column_stack([
        ordinal_prediction(scores[:, i], trait) for i, trait in enumerate(RUBRIC['traits'])
    ])
    rows = []
    for j, i in enumerate(indices):
        source = bundle.rows[i]
        row = {**keys, 'id_jawaban': source['row_id'], 'baris_excel': source['source_row'],
               'no_soal': source['response_number'], 'kelompok': int(bundle.groups[i])}
        for col, (name, _, _, _) in enumerate(TRAITS):
            actual, raw = bundle.y[i, col], scores[j, col]
            row.update({f'{name}_label': int(actual), f'{name}_prediksi_mentah': float(raw),
                        f'{name}_prediksi_kategori': int(categories[j, col]),
                        f'{name}_error_abs': float(abs(raw - actual))})
        rows.append(row)
    return rows


def history_rows(info, **keys):
    return [{**keys, **{k: v for k, v in row.items() if k != 'validation'},
             'validation_RMSE': row.get('validation', {}).get('RMSE')}
            for row in info['history']]


def export_results(path, bundle, manifest, cfg, cv_summary, folds, final_info, prediction, inspection=None):
    cv_fold, oof_rows, history, indices, predictions = [], [], [], [], []
    for result in folds:
        fold = result['identity']['fold']
        valid = result['identity']['validation']
        info = result['training']
        cv_fold.extend(trait_rows(result['metrics'], fold=fold,
                                  epoch_terpilih=info['best_epoch'], n=len(valid)))
        oof_rows.extend(prediction_rows(bundle, valid, result['prediction'], fold=fold))
        history.extend(history_rows(info, fold=fold))
        indices.extend(valid)
        predictions.extend(result['prediction'])
    cv_oof = trait_rows(metrics(bundle.y[indices], predictions), n=len(indices))

    test = manifest['test']
    test_metrics = trait_rows(metrics(bundle.y[test], prediction), n=len(test))
    history.extend(history_rows(final_info, fold='refit'))
    evaluation = {
        'Ringkasan CV': pd.DataFrame([cv_summary]),
        'CV per Fold dan Trait': pd.DataFrame(cv_fold),
        'CV OOF per Trait': pd.DataFrame(cv_oof),
        'Prediksi OOF': pd.DataFrame(oof_rows),
        'Test per Trait': pd.DataFrame(test_metrics),
        'Prediksi Test': pd.DataFrame(prediction_rows(bundle, test, prediction)),
        'Skor Terendah': pd.DataFrame(
            lowest_score_rows(bundle, indices, predictions, 'OOF')
            + lowest_score_rows(bundle, test, prediction, 'test')
        ),
    }
    evaluation['Matriks Metrik CV dan Test'] = pd.DataFrame([
        {'bagian': stage, 'trait': row['trait'], 'n': row.get('n'),
         'RMSE': row['RMSE'], 'Pearson': row['Pearson'],
         'QWK': row['QWK'], 'Gwet AC2': row['Gwet AC2']}
        for stage, rows in [('CV OOF', cv_oof), ('Test', test_metrics)]
        for row in rows
    ])
    for stage, ids, values in [('OOF', indices, predictions), ('Test', test, prediction)]:
        for column, trait in enumerate(RUBRIC['traits']):
            categories = trait['categories']
            actual = bundle.y[ids, column].astype(int)
            predicted = ordinal_prediction(np.asarray(values)[:, column], trait)
            matrix = np.asarray([
                [int(np.sum((actual == a) & (predicted == p))) for p in categories]
                for a in categories
            ])
            evaluation[f'CM_{stage}_{trait["name"]}'] = pd.DataFrame(
                matrix, index=[f'aktual_{v}' for v in categories],
                columns=[f'prediksi_{v}' for v in categories]
            ).reset_index(names='aktual')
    groups = {
        '01_Data_dan_Tokenisasi': {},
        '02_Vektor_Jawaban': {},
        '03_Vektor_Kata_dan_Kalimat': {},
        '04_Prediksi_dan_Evaluasi': evaluation,
        '05_Riwayat_Pelatihan': {'Riwayat Training': pd.DataFrame(history)},
    }
    for name, frame in (inspection or {}).items():
        if name.startswith('Kata'):
            workbook = '03_Vektor_Kata_dan_Kalimat'
        elif name.startswith('Final_') or name in ('RoBERTa pretrained', 'BGE frozen'):
            workbook = '02_Vektor_Jawaban'
        elif name in ('Konfigurasi', 'Arsitektur'):
            workbook = '05_Riwayat_Pelatihan'
        else:
            workbook = '01_Data_dan_Tokenisasi'
        groups[workbook][name] = frame
    return export_excel_zip(path, groups, {
        **bundle.summary, 'eksperimen': 'Reset_E17E_TPE_Concat', 'dataset_version': cfg['dataset_version'],
        'development_rows': len(manifest['development']), 'test_rows': len(test),
        'split_sha256': manifest['split_sha256'], 'notebook_code_id': NOTEBOOK_CODE_ID,
        'test_dipakai_untuk_seleksi_epoch': False, 'final_config': cfg,
        'kriteria_early_stopping': 'Akar rata-rata kuadrat error seluruh skor validasi dalam satuan asli.',
    })


## 18 · Ruang pencarian, split, dan kontrak resume

In [ ]:
# Batas ini adalah rancangan eksplorasi lokal; baseline tetap dicatat terpisah.
SEARCH_SPACE = {
    'encoder_lr': {'low': 1e-6, 'high': 2e-5, 'log': True},
    'aux_lr': {'low': 3e-5, 'high': 1e-3, 'log': True},
    'mlp_dropout': {'low': 0.05, 'high': 0.40, 'step': 0.05},
    'weight_decay': [0.0, 0.001, 0.005, 0.01, 0.03, 0.05],
    'warmup_ratio': [0.03, 0.05, 0.10, 0.15],
}
BASELINE = dict(encoder_lr=5e-6, aux_lr=1e-4, mlp_dropout=0.1,
                weight_decay=0.01, warmup_ratio=0.10)
# Titik awal yang ditetapkan sebelum melihat hasil E17E, bukan hasil prediksi.
ANCHOR_VALUES = [
    (5e-6,1e-4,.10,.010,.10), (3e-6,1e-4,.10,.010,.10),
    (7e-6,1e-4,.10,.010,.10), (1e-5,1e-4,.10,.010,.10),
    (5e-6,5e-5,.10,.010,.10), (5e-6,3e-4,.10,.010,.10),
    (5e-6,1e-4,.20,.010,.10), (5e-6,1e-4,.30,.010,.10),
    (5e-6,1e-4,.10,.001,.10), (5e-6,1e-4,.10,.050,.10),
    (5e-6,1e-4,.10,.010,.05), (5e-6,1e-4,.10,.010,.15),
    (2e-6,5e-5,.20,.030,.10), (3e-6,3e-4,.25,.010,.05),
    (1e-5,5e-4,.30,.030,.15), (2e-5,1e-3,.40,.050,.10),
]
ANCHORS = [dict(zip(BASELINE, values)) for values in ANCHOR_VALUES]

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     allow_nan=False).encode()).hexdigest()

def parameter_key(params):
    # Canonicalize decimal steps (0.15000000002 and 0.15 are one setting).
    return stable_hash({k: float(format(float(v), '.12g')) for k,v in params.items()})

def macro_qwk(per_trait):
    values=[v['QWK'] for v in per_trait.values()]
    if len(values)!=len(TRAITS) or any(v is None or not math.isfinite(v) for v in values):
        return None  # Jangan diam-diam membuang trait yang metriknya undefined.
    return float(np.mean(values))

def fit_config_hash(value):
    ignored = {'dataset', 'output', 'runtime', 'trace_position', 'vector_source'}
    return stable_hash({'code': HPO_CODE_SHA256,
                        'config': {k:v for k,v in value.items() if k not in ignored}})

def candidate_config(base_cfg, params, seed=42):
    result = {**base_cfg, 'seed': int(seed),
              'encoder_lr': params['encoder_lr'], 's3_encoder_lr': params['encoder_lr'],
              'head_lr': params['aux_lr'], 's3_mlp_lr': params['aux_lr'],
              'projection_lr': params['aux_lr'], 's3_layers_lr': params['aux_lr'],
              's3_mlp_dropout': params['mlp_dropout'],
              'weight_decay': params['weight_decay'], 'warmup_ratio': params['warmup_ratio']}
    return config(result)

def make_search_partition(bundle, manifest):
    # Split development saja. Tidak membaca nilai target atau memilih dari skor.
    dev = np.asarray(manifest['development'], dtype=int)
    a,b = next(GroupShuffleSplit(n_splits=1, test_size=.2,
                                random_state=HPO_SPLIT_SEED).split(dev, groups=bundle.groups[dev]))
    train, valid = dev[a].tolist(), dev[b].tolist()
    if set(bundle.groups[train]) & set(bundle.groups[valid]):
        raise ValueError('Kelompok siswa bocor pada pencarian.')
    if (set(train)|set(valid)) != set(dev) or set(train)&set(valid):
        raise ValueError('Partisi pencarian tidak tepat membagi development.')
    if (set(train)|set(valid)) & set(manifest['test']):
        raise ValueError('Test tidak boleh masuk pencarian.')
    return {'train':train, 'validation':valid, 'seed':HPO_SPLIT_SEED,
            'selection_uses_labels':False}

search_partition = make_search_partition(bundle, manifest)
STUDY_CONTRACT = {
    'code_sha256': HPO_CODE_SHA256, 'source_sha256': bundle.source_sha256,
    'main_split': manifest['split_sha256'], 'search_partition':search_partition,
    'base_fit':fit_config_hash(cfg), 'search_space':SEARCH_SPACE, 'anchors':ANCHORS,
    'objective':'minimum pooled validation RMSE in original trait units',
    'seed':HPO_SEED, 'top_k':TOP_K, 'stability_seeds':list(STABILITY_SEEDS),
    'sampler':'TPE multivariate; per-attempt seed=HPO_SEED+trial-number',
    'pruner':'Median; 16 completed startup trials; earliest epoch5; anchors exempt',
    'versions':{'optuna':version('optuna'), **runtime_information()},
    'scheduler_policy':'linear; horizon=max epochs of each fit; refit horizon=refit duration',
}
STUDY_ID = stable_hash(STUDY_CONTRACT)
STUDY_DIR = OUTPUT_ROOT / 'studies' / STUDY_ID[:20]
STUDY_DIR.mkdir(parents=True, exist_ok=True)
contract_path = STUDY_DIR/'contract.json'
if contract_path.exists() and stable_hash(json.loads(contract_path.read_text())) != STUDY_ID:
    raise ValueError('Kontrak study berbeda; jangan mencampur hasil.')
dump(contract_path, STUDY_CONTRACT)
print('Study:', STUDY_DIR)
print('Pencarian:',len(search_partition['train']),'train,',len(search_partition['validation']),'validation; test tidak dipakai.')
display(pd.DataFrame(ANCHORS).rename_axis('anchor').reset_index())


## 19 · Study Optuna dan snapshot Drive

In [ ]:
# SQLite bekerja pada disk lokal. Drive hanya menyimpan snapshot konsisten.
LOCAL_STUDY_DIR = Path(tempfile.mkdtemp(prefix='aes_e17e_'))
LOCAL_DB = LOCAL_STUDY_DIR/'study.sqlite3'
SNAPSHOT_POINTER = STUDY_DIR/'snapshot.json'

def restore_database():
    if SNAPSHOT_POINTER.exists():
        proof=json.loads(SNAPSHOT_POINTER.read_text())
        saved=STUDY_DIR/proof['file']
        if proof['study_id']!=STUDY_ID or digest(saved)!=proof['sha256']:
            raise ValueError('Snapshot study tidak cocok atau rusak.')
        shutil.copyfile(saved, LOCAL_DB)
        with sqlite3.connect(LOCAL_DB) as db:
            if db.execute('PRAGMA integrity_check').fetchone()[0]!='ok':
                raise ValueError('SQLite integrity check gagal.')

def backup_database():
    if not LOCAL_DB.exists(): return
    temporary=STUDY_DIR/'study.pending.sqlite3'
    temporary.unlink(missing_ok=True)
    with sqlite3.connect(LOCAL_DB) as src, sqlite3.connect(temporary) as dst:
        src.backup(dst)
        if dst.execute('PRAGMA integrity_check').fetchone()[0]!='ok':
            raise IOError('Snapshot SQLite tidak konsisten.')
    checksum=digest(temporary)
    target=STUDY_DIR/f'study_{checksum}.sqlite3'
    temporary.replace(target)
    previous=json.loads(SNAPSHOT_POINTER.read_text()) if SNAPSHOT_POINTER.exists() else None
    dump(SNAPSHOT_POINTER, {'study_id':STUDY_ID,'file':target.name,'sha256':checksum})
    # Hapus snapshot lama hanya sesudah pointer baru tersimpan.
    if previous and previous['file']!=target.name:
        (STUDY_DIR/previous['file']).unlink(missing_ok=True)

restore_database()
study = optuna.create_study(
    study_name='E17E_'+STUDY_ID[:20], direction='minimize',
    storage='sqlite:///'+str(LOCAL_DB), load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=HPO_SEED, n_startup_trials=16, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=16, n_warmup_steps=5,
                                     interval_steps=1, n_min_trials=5))
if not study.trials:
    study.set_user_attr('contract_sha256', STUDY_ID)
    for number,params in enumerate(ANCHORS,1):
        study.enqueue_trial(params, user_attrs={'anchor':number})
elif study.user_attrs.get('contract_sha256') != STUDY_ID:
    raise ValueError('Database tidak cocok dengan kontrak study.')

# Hanya satu runtime boleh aktif. RUNNING pada snapshot berarti fit terputus.
# Parameter lengkap diretry dari awal; weights/optimizer setengah fit tidak direuse.
recovered=set(study.user_attrs.get('recovered_attempts',[]))
for frozen in list(study.trials):
    if frozen.state==TrialState.RUNNING:
        study.tell(frozen.number, state=TrialState.FAIL)
    if frozen.state in (TrialState.RUNNING,TrialState.FAIL) and frozen.number not in recovered:
        if set(frozen.params)==set(BASELINE):
            study.enqueue_trial(frozen.params, user_attrs={
                **frozen.user_attrs, 'retry_of':frozen.number})
        recovered.add(frozen.number)
study.set_user_attr('recovered_attempts',sorted(recovered))
backup_database()

def finished_trials():
    return [t for t in study.trials if t.state in (TrialState.COMPLETE, TrialState.PRUNED)]

def trial_table():
    return pd.DataFrame([{
        'trial':t.number, 'status':t.state.name, 'RMSE':t.value,
        **t.params, 'anchor':t.user_attrs.get('anchor'),
        'retry_of':t.user_attrs.get('retry_of'),
        'best_epoch':t.user_attrs.get('best_epoch'),
        'elapsed_seconds':t.user_attrs.get('elapsed_seconds'),
        'scope':'development_search_holdout_selection_only',
    } for t in study.trials])

def search_history_table():
    result=[]
    for path in sorted((STUDY_DIR/'trials').glob('trial_*.json')):
        saved=json.loads(path.read_text())
        for row in saved['history']:
            result.append({'trial':saved['trial'], **{k:v for k,v in row.items() if k!='validation'},
                           'validation_RMSE':row.get('validation',{}).get('RMSE')})
    return pd.DataFrame(result)

def export_search_report():
    path=STUDY_DIR/'06_Pencarian_Hyperparameter.xlsx'
    write_excel(path, {'Trials':trial_table(), 'Riwayat Trial':search_history_table(),
                      'Ruang Pencarian':pd.DataFrame([{'parameter':k,'rentang':v} for k,v in SEARCH_SPACE.items()])},
                {'study_id':STUDY_ID, 'scope':'development_only_hyperparameter_selection',
                 'budget':HPO_TRIALS,'finished':len(finished_trials()),'test_used':False})
    print('Laporan pencarian:',path)


## 20 · Satu trial dan pencarian adaptif

In [ ]:
def suggest_parameters(trial):
    return {
        'encoder_lr':trial.suggest_float('encoder_lr',**SEARCH_SPACE['encoder_lr']),
        'aux_lr':trial.suggest_float('aux_lr',**SEARCH_SPACE['aux_lr']),
        'mlp_dropout':trial.suggest_float('mlp_dropout',**SEARCH_SPACE['mlp_dropout']),
        'weight_decay':trial.suggest_categorical('weight_decay',SEARCH_SPACE['weight_decay']),
        'warmup_ratio':trial.suggest_categorical('warmup_ratio',SEARCH_SPACE['warmup_ratio']),
    }

def objective(trial):
    params=suggest_parameters(trial)
    trial_cfg=candidate_config(cfg,params,TRAIN_SEED)
    history_path=STUDY_DIR/'trials'/f'trial_{trial.number:04d}.json'
    started=time.perf_counter()
    model=engine=None

    def report_epoch(epoch,row,history):
        # Best-so-far RMSE dibandingkan pada epoch yang sama di seluruh trial.
        best=min(r['validation']['RMSE'] for r in history)
        trial.report(best, step=epoch)
        dump(history_path, {'trial':trial.number,'params':params,'history':history})
        backup_database()
        return not trial.user_attrs.get('anchor') and trial.should_prune()

    try:
        model,engine,info=fit('S3',search_partition['train'],search_partition['validation'],
                              bundle,bge,trial_cfg,EPOCHS,TRAIN_SEED,epoch_callback=report_epoch)
        trial.set_user_attr('best_epoch',info['best_epoch'])
        trial.set_user_attr('elapsed_seconds',time.perf_counter()-started)
        dump(history_path,{'trial':trial.number,'params':params,'history':info['history'],
                           'training':info})
        if info['pruned']:
            raise optuna.TrialPruned('Pruning dari metrik development; test tidak dibaca.')
        # Verifikasi bobot best epoch yang dipulihkan, bukan model epoch terakhir.
        prediction=engine.predict(search_partition['validation'])
        value=metrics(bundle.y[search_partition['validation']],prediction)['RMSE']
        if not math.isclose(value,info['best_validation_rmse'],abs_tol=1e-6):
            raise ValueError('Prediksi checkpoint trial tidak cocok dengan best epoch.')
        return value
    except BaseException as error:
        # Bebaskan tensor dari frame fit jika CUDA OOM/interrupt terjadi.
        traceback.clear_frames(error.__traceback__)
        raise
    finally:
        model=engine=None
        release()

def run_search():
    if HPO_TRIALS<16: raise ValueError('Budget minimal 16 agar semua anchor teruji.')
    while len(finished_trials())<HPO_TRIALS:
        # Reinitialisasi seed per attempt: lanjut dari snapshot tidak perlu pickle RNG.
        next_number=len(study.trials)
        waiting=[t for t in study.trials if t.state==TrialState.WAITING]
        if waiting: next_number=min(t.number for t in waiting)
        study.sampler=optuna.samplers.TPESampler(
            seed=HPO_SEED+next_number,n_startup_trials=16,multivariate=True)
        try:
            study.optimize(objective,n_trials=1,n_jobs=1,gc_after_trial=True)
        finally:
            backup_database()
            export_search_report()
        print(f'Selesai {len(finished_trials())}/{HPO_TRIALS} trial COMPLETE/PRUNED.',flush=True)
    display(trial_table().sort_values('RMSE',na_position='last'))


## 21 · Konfirmasi CV dan stabilitas seed

In [ ]:
def search_fingerprint():
    # Termasuk WAITING/FAIL supaya hasil seleksi tidak diam-diam memakai study berubah.
    return stable_hash([{'number':t.number,'state':t.state.name,'params':t.params,'value':t.value}
                        for t in study.trials])

def candidate_plan():
    if len(finished_trials())<HPO_TRIALS:
        raise RuntimeError('Selesaikan budget search sebelum konfirmasi/final.')
    selected=[{'candidate':'baseline','params':BASELINE,'trial':None}]
    seen={parameter_key(BASELINE)}
    completed=sorted([t for t in study.trials if t.state==TrialState.COMPLETE],
                     key=lambda t:(t.value,t.number))
    for trial in completed:
        key=parameter_key(trial.params)
        if key in seen: continue
        selected.append({'candidate':key[:12],'params':trial.params,'trial':trial.number})
        seen.add(key)
        if len(selected)==TOP_K+1: break
    if len(selected)<TOP_K+1:
        raise RuntimeError('Belum cukup kandidat COMPLETE unik. Tambah budget search.')
    return selected

def confirmation_root():
    return STUDY_DIR/'confirmation'/search_fingerprint()[:20]

def candidate_cv(entry,seed,root):
    local_cfg=candidate_config(cfg,entry['params'],seed)
    folder=root/entry['candidate']/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    summary,folds=run_cv(bundle,manifest,bge,local_cfg,folder,'S3')
    ids=[i for r in folds for i in r['identity']['validation']]
    pred=[p for r in folds for p in r['prediction']]
    pooled=metrics(bundle.y[ids],pred)
    row={'candidate':entry['candidate'],'seed':seed,'trial':entry['trial'],
         **entry['params'],**summary,'pooled_OOF_RMSE':pooled['RMSE'],
         'macro_QWK':macro_qwk(pooled['traits'])}
    dump(folder/'cv_summary.json',{'row':row,'scope':'CV used for hyperparameter selection',
                                  'per_trait':pooled['traits']})
    return row,summary,folds

def run_confirmation():
    if tuple(STABILITY_SEEDS)!=(TRAIN_SEED,43,44):
        raise ValueError('Seed protokol ini harus 42,43,44; ubah protokol study jika perlu.')
    plan=candidate_plan(); root=confirmation_root();root.mkdir(parents=True,exist_ok=True)
    dump(root/'plan.json',{'study':STUDY_ID,'search_fingerprint':search_fingerprint(),
                           'candidates':plan,'primary_seed':TRAIN_SEED})
    ranking=[]; per_fold=[]
    for entry in plan:
        row,_,folds=candidate_cv(entry,TRAIN_SEED,root)
        ranking.append(row)
        per_fold.extend({'candidate':entry['candidate'],'seed':TRAIN_SEED,
                         'fold':r['identity']['fold'],'RMSE':r['metrics']['RMSE'],
                         'best_epoch':r['training']['best_epoch']} for r in folds)
        write_excel(root/'07_Konfirmasi_CV.xlsx',{
            'Ranking CV':pd.DataFrame(ranking),'Fold CV':pd.DataFrame(per_fold)},
            {'study_id':STUDY_ID,'test_used':False,'selection_metric':'mean_cv_RMSE',
             'scope':'selection CV, not unbiased nested-CV'})
    # Ranking berdasarkan mean pooled RMSE antarfold; baseline selalu eligible.
    best=min(ranking,key=lambda r:(r['mean_cv_RMSE'],r['candidate']!='baseline',r['candidate']))
    winner=next(e for e in plan if e['candidate']==best['candidate'])
    stability=[]
    compare=[plan[0]]+([winner] if winner['candidate']!='baseline' else [])
    for entry in compare:
        for seed in STABILITY_SEEDS:
            row,_,_=candidate_cv(entry,seed,root)
            stability.append(row)
            write_excel(root/'08_Stabilitas_Seed.xlsx',{'Per Seed':pd.DataFrame(stability)},
                        {'study_id':STUDY_ID,'scope':'development diagnostic; winner fixed by primary CV',
                         'test_used':False,'seeds':STABILITY_SEEDS})
    # Stabilitas dilaporkan, tidak memindahkan pemilihan parameter ke test.
    cv_summary,folds=run_cv(bundle,manifest,bge,candidate_config(cfg,winner['params']),
                          root/winner['candidate']/f'seed_{TRAIN_SEED}','S3')
    proof_files={str(p.relative_to(root)):digest(p) for p in root.glob('*/seed_*/fold_*.json')}
    seal={'study_id':STUDY_ID,'search_fingerprint':search_fingerprint(),
          'search_budget':HPO_TRIALS,'winner':winner,'cv_summary':cv_summary,
          'ranking':ranking,'stability':stability,'files_sha256':proof_files,
          'selection_rule':'minimum mean_cv_RMSE, primary seed42; baseline eligible',
          'test_used_for_selection':False}
    dump(root/'selection_locked.json',{'payload':seal,'sha256':stable_hash(seal)})
    display(pd.DataFrame(ranking).sort_values('mean_cv_RMSE'))
    display(pd.DataFrame(stability))
    print('Konfigurasi dikunci:',winner,'; mode final dapat dijalankan berikutnya.')

def load_locked_selection():
    candidate_plan()  # Konfirmasi jumlah trial dan kandidat unik.
    root=confirmation_root();path=root/'selection_locked.json'
    if not path.exists(): raise RuntimeError('Jalankan STAGE=confirm sampai selesai dahulu.')
    wrapped=json.loads(path.read_text()); seal=wrapped['payload']
    if (stable_hash(seal)!=wrapped['sha256'] or seal['study_id']!=STUDY_ID
        or seal['search_fingerprint']!=search_fingerprint()
        or seal['search_budget']!=HPO_TRIALS or seal['test_used_for_selection']):
        raise ValueError('Seleksi terkunci tidak cocok dengan study saat ini.')
    for name,checksum in seal['files_sha256'].items():
        if digest(root/name)!=checksum: raise ValueError('Hasil konfirmasi berubah: '+name)
    return root,seal


## 22 · Konfigurasi terkunci, refit, dan test

In [ ]:
def run_final():
    # Pintu masuk test hanya ada di fungsi ini dan hanya dipanggil pada mode final.
    root,seal=load_locked_selection()
    entry=seal['winner']; final_cfg=candidate_config(cfg,entry['params'],TRAIN_SEED)
    cv_summary,folds=run_cv(bundle,manifest,bge,final_cfg,
                          root/entry['candidate']/f'seed_{TRAIN_SEED}','S3')
    if cv_summary!=seal['cv_summary']:
        raise ValueError('Ringkasan CV berbeda dari seleksi terkunci.')
    final_folder=root/'final'/fit_config_hash(final_cfg)[:20]
    final_folder.mkdir(parents=True,exist_ok=True)
    dump(final_folder/'locked_config.json',{'config':final_cfg,'selection':seal})
    refit=refit_checkpoint(cv_summary['epoch_refit'],bundle,manifest,bge,final_cfg,final_folder,'S3')
    prediction=evaluate_test(refit,bundle,manifest,bge,final_cfg,final_folder)
    stats=metrics(bundle.y[manifest['test']],prediction)
    display(pd.DataFrame(trait_rows(stats,n=len(manifest['test']))))
    display(pd.DataFrame([{'bagian':'test','pooled_RMSE':stats['RMSE'],
                          'macro_QWK':macro_qwk(stats['traits'])}]))
    extra={
        'Konfigurasi':pd.DataFrame([{'parameter':k,'nilai':v} for k,v in final_cfg.items()]),
        'Arsitektur':pd.DataFrame([{'bagian':k,'dimensi':v} for k,v in
            [('RoBERTa',768),('BGE frozen',1024),('Projection',256),('Concat',1024),('Hidden',256),('Trait',4)]]),
    }
    # Tambahan tabel HPO masuk workbook05 lewat wrapper ekspor transparan.
    hpo_sheets={'HPO Trials':trial_table(), 'HPO Riwayat':search_history_table(),
                'HPO Konfirmasi':pd.DataFrame(seal['ranking']),
                'HPO Stabilitas':pd.DataFrame(seal['stability'])}
    original_export=globals()['export_excel_zip']
    def export_with_hpo(path,groups,metadata,max_bytes=30_000_000):
        groups['05_Riwayat_Pelatihan'].update(hpo_sheets)
        metadata={**metadata,'hpo_study_id':STUDY_ID,'selection_scope':'development search + selection CV',
                  'cv_is_post_selection':True,'test_previously_inspected_in_research':True}
        return original_export(path,groups,metadata,max_bytes)
    report_path=final_folder/'Hasil_AES_Reset_E17E_Ringkas_04_05.zip'
    try:
        globals()['export_excel_zip']=export_with_hpo
        inventory=export_results(report_path,bundle,manifest,{**final_cfg,'scenario':'S3'},
                                 cv_summary,folds,refit['training'],prediction,inspection=extra)
    finally:
        globals()['export_excel_zip']=original_export
    # Gate laporan: baca kembali ZIP/Excel dan cocokkan prediksi dengan checkpoint test.
    verify_final_report(report_path, prediction, stats)
    dump(final_folder/'test_metrics.json',{'metrics':stats,'report_sha256':digest(report_path),
                                         'checkpoint_sha256':refit['model_sha256']})
    display(pd.DataFrame(inventory)[['file','bytes']])
    print('Hasil privat Drive:',report_path)
    if 'google.colab' in sys.modules:
        from google.colab import files
        from ipywidgets import Button
        button=Button(description='Unduh hasil E17E')
        button.on_click(lambda _:files.download(str(report_path)))
        display(button)

def verify_final_report(path,prediction,stats):
    import io
    collected=[]; trait_stats=[]; saw_hpo=False
    with zipfile.ZipFile(path) as archive:
        if archive.testzip() is not None: raise IOError('ZIP korup.')
        proof=json.loads(archive.read(next(n for n in archive.namelist() if n.endswith('/manifest.json'))))
        for item in proof['workbooks']:
            raw=archive.read(next(n for n in archive.namelist() if n.endswith('/'+item['file'])))
            if hashlib.sha256(raw).hexdigest()!=item['sha256']: raise IOError('Hash workbook tidak cocok.')
            book=load_workbook(io.BytesIO(raw),read_only=True,data_only=True)
            try:
                if 'Prediksi Test' in book.sheetnames:
                    data=iter(book['Prediksi Test'].values);header=next(data)
                    collected.extend(dict(zip(header,row)) for row in data)
                if 'Test per Trait' in book.sheetnames:
                    data=iter(book['Test per Trait'].values);header=next(data)
                    trait_stats.extend(dict(zip(header,row)) for row in data)
                saw_hpo = saw_hpo or 'HPO Trials' in book.sheetnames
            finally: book.close()
    expected_ids=[bundle.rows[i]['row_id'] for i in manifest['test']]
    if [r['id_jawaban'] for r in collected]!=expected_ids or not saw_hpo:
        raise ValueError('ID/jumlah prediksi atau tabel HPO laporan tidak cocok.')
    values=np.array([[r[t[0]+'_prediksi_mentah'] for t in TRAITS] for r in collected])
    if not np.allclose(values,prediction,rtol=0,atol=1e-12):
        raise ValueError('Prediksi Excel berbeda dari hasil test tersimpan.')
    for row in trait_stats:
        if not math.isclose(row['RMSE'],stats['traits'][row['trait']]['RMSE'],abs_tol=1e-10):
            raise ValueError('RMSE Excel berbeda dari hasil perhitungan.')
    if len(trait_stats)!=4: raise ValueError('Empat trait harus lengkap.')


## 23 · Jalankan mode terpilih

In [ ]:
# Peluncur konfirmasi dan final memakai kontrak training E17E asli.
if STAGE == 'search':
    run_search()
elif STAGE == 'confirm':
    run_confirmation()
elif STAGE == 'final':
    run_final()
elif STAGE == 'confirm_final':
    if not SNAPSHOT_POINTER.exists() or len(finished_trials()) < HPO_TRIALS:
        raise RuntimeError(
            f'Study belum siap: {STUDY_DIR}. '
            f'Ditemukan {len(finished_trials())}/{HPO_TRIALS} trial selesai. '
            'Gunakan Drive dan versi runtime study pencarian yang sama.'
        )
    print('Study konfirmasi:', STUDY_DIR, flush=True)
    print('Trial selesai:', len(finished_trials()), flush=True)
    run_confirmation()
    # run_final verifies selection lock and fold checksums before refit/test.
    run_final()
print('Mode selesai:', STAGE)
print('Folder study privat:', STUDY_DIR)
